In [ ]:
import platform
import sys
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib

print(f"Python       {sys.version.split()[0]} on {platform.system()}")
for lib in (np, pd, scipy, sklearn, matplotlib):
    print(f"{lib.__name__:<12} {lib.__version__}")

# Optional extras. The notebooks fall back gracefully if these are absent.
for name in ("seaborn", "statsmodels"):
    try:
        mod = __import__(name)
        print(f"{name:<12} {mod.__version__}")
    except ImportError:
        print(f"{name:<12} not installed (optional)")

Python       3.11.16 on Windows
numpy        2.4.6
pandas       3.0.5
scipy        1.17.1
sklearn      1.9.0
matplotlib   3.11.1
seaborn      0.13.2
statsmodels  0.15.0


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path.cwd()
# data files may sit in ./data or right next to the notebook
DATA = next((p for p in (ROOT / "data", ROOT) if (p / "anggota.csv").exists()), ROOT / "data")
pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 30)
print("data folder:", DATA)

data folder: c:\Users\ahmad\dev-resources\S2\data-science\project-pasca-uts\data


In [ ]:
SENTINELS = ["", "NA", "N/A", "-", "null", "NULL", "unknown", "-999"]

rawDesa = pd.read_excel(
    DATA / "desa.xlsx",
    dtype={"id": "string"},             # preserve leading zeros
    na_values=SENTINELS,                # declare the folk sentinels
    keep_default_na=True,
    parse_dates=["dihitung_pada"],
    # dayfirst=True,                      # 03/04/2023 is 3 April
)
print("shape:", rawDesa.shape)
print(rawDesa.dtypes.to_string())
rawDesa.head(5)

shape: (79842, 33)
kode                               int64
nama                                 str
tipe                                 str
kecamatan                            str
kabupaten                            str
provinsi                             str
luas_desa                        float64
jumlah_persil                      int64
luas_persil                        int64
hak_milik_jumlah                   int64
hak_milik_luas                     int64
hgb_jumlah                         int64
hgb_luas                           int64
hgu_jumlah                         int64
hgu_luas                           int64
hak_pakai_jumlah                   int64
hak_pakai_luas                     int64
hak_pengelolaan_jumlah             int64
hak_pengelolaan_luas               int64
hak_wakaf_jumlah                   int64
hak_wakaf_luas                     int64
kosong_jumlah                      int64
kosong_luas                        int64
lainnya_jumlah                     int

,kode,nama,tipe,kecamatan,kabupaten,provinsi,luas_desa,jumlah_persil,luas_persil,hak_milik_jumlah,hak_milik_luas,hgb_jumlah,hgb_luas,hgu_jumlah,hgu_luas,...,hak_pengelolaan_luas,hak_wakaf_jumlah,hak_wakaf_luas,kosong_jumlah,kosong_luas,lainnya_jumlah,lainnya_luas,tahun_min,tahun_median,tahun_max,rdtr_luas,banjir_rendah,banjir_sedang,banjir_tinggi,dihitung_pada
0,1010101,KOTA BARU,Kelurahan,Kuta Alam,Banda Aceh,Aceh,593698.7,313,679003,219,95859,1,6125,0,0,...,0,0,0,46,181137,0,0,1972.0,2006.0,2026.0,593686.4,22987.2,130725.3,439986.2,2026-10-17 15:12:38
1,1010102,BANDAR BARU,Kelurahan,Kuta Alam,Banda Aceh,Aceh,1139905.6,797,1016060,668,294883,20,8270,0,0,...,0,5,1299,43,84498,0,0,1972.0,2004.0,2026.0,1139905.6,0.0,815590.7,324315.3,2026-10-17 15:12:38
2,1010103,BEURAWE,Kelurahan,Kuta Alam,Banda Aceh,Aceh,799736.9,1601,737670,1455,455608,19,27576,0,0,...,0,2,472,93,190819,0,0,1974.0,2012.0,2026.0,799736.9,0.0,350871.9,448864.9,2026-10-17 15:12:38
3,1010104,KUTA ALAM,Kelurahan,Kuta Alam,Banda Aceh,Aceh,511125.6,576,553700,417,86916,38,25835,0,0,...,2572,2,199,76,132973,0,0,1964.0,2003.0,2026.0,511112.5,0.0,501677.5,9448.1,2026-10-17 15:12:38
4,1010105,KEURAMAT,Kelurahan,Kuta Alam,Banda Aceh,Aceh,326105.0,932,336401,792,174164,18,12373,0,0,...,0,3,1309,79,52816,0,0,1974.0,2001.0,2026.0,326105.0,0.0,326105.0,0.0,2026-10-17 15:12:38


In [ ]:
import pandas as pd

def profile(df):
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(dropna=True),
        "example": [
            df[c].dropna().sample(n=min(2, df[c].notna().sum())).tolist() 
            if df[c].notna().any() else []
            for c in df.columns
        ],
    })
    return out.sort_values("pct_missing", ascending=False)

In [ ]:
profile(rawDesa)

,dtype,n_missing,pct_missing,n_unique,example
tahun_max,float64,2670,3.3,61,"[2025.0, 2018.0]"
tahun_median,float64,2670,3.3,62,"[2021.0, 2019.0]"
tahun_min,float64,2670,3.3,67,"[2012.0, 2011.0]"
kode,int64,0,0.0,79842,"[10080302, 2040316]"
nama,str,1,0.0,63428,"[Larete, Dorolamo]"
provinsi,str,0,0.0,34,"[Nusa Tenggara Barat, Nusa Tenggara Timur]"
tipe,str,0,0.0,2,"[Desa, Kelurahan]"
kecamatan,str,1,0.0,6732,"[Dumoga Tengah, Sayung]"
kabupaten,str,0,0.0,487,"[Malaka, GORONTALO]"
hak_milik_jumlah,int64,0,0.0,5826,"[1158, 323]"


# Relationship inference on `rawDesa`

Continuation of the notebook. Plan: **(1)** structural relations, **(2)** size-neutral features, **(3)** robust prep.
Steps 4-8 (association matrices, latent structure, within-group checks, ...) will build on the objects created here.

## Step 1: Structural relations

**Goal:** find the relationships that exist *by construction* (identities, inequalities, null patterns, code hierarchy) before looking for statistical ones.

**Why first:** columns that are computed from each other correlate perfectly, and that is not insight. Knowing them up front tells us which columns are redundant and which "discoveries" are fake.

In [ ]:
# --- Working copy + column groups -----------------------------------------------------
desa = rawDesa.copy()      # rawDesa stays untouched; everything below works on `desa`

RIGHTS = ["hak_milik", "hgb", "hgu", "hak_pakai", "hak_pengelolaan", "hak_wakaf", "kosong", "lainnya"]
CNT = [f"{r}_jumlah" for r in RIGHTS]      # parcel counts per right
ARE = [f"{r}_luas" for r in RIGHTS]        # parcel areas per right
FLOOD = ["banjir_rendah", "banjir_sedang", "banjir_tinggi"]
YEARS = ["tahun_min", "tahun_median", "tahun_max"]

# `kode` was read as an integer, so its leading zero is gone (1010101 is really 01010101).
# Restore the 8-digit code: PP (province) KK (kab/kota) CC (kecamatan) DD (village)
desa["kode8"] = desa["kode"].astype("int64").astype(str).str.zfill(8)
desa["prov_code"] = desa["kode8"].str[:2]
desa["kab_code"] = desa["kode8"].str[:4]
desa["kec_code"] = desa["kode8"].str[:6]

print("kode length before padding:", desa["kode"].astype(str).str.len().value_counts().to_dict())
print("kode length after padding :", desa["kode8"].str.len().value_counts().to_dict())
desa[["kode", "kode8", "prov_code", "kab_code", "kec_code", "nama", "kabupaten", "provinsi"]].head()

In [ ]:
# --- Rule book: every structural relation we test gets one row -------------------------
RULES = []     # re-running this cell starts a fresh rule book

def add_rule(name, kind, violated, scope=None, note=""):
    """violated: boolean Series, True where the rule is broken.
       scope: boolean Series of the rows the rule applies to (default: all rows)."""
    scope = pd.Series(True, index=desa.index) if scope is None else scope
    n, v = int(scope.sum()), int((violated & scope).sum())
    RULES.append({"rule": name, "kind": kind, "n_checked": n, "n_violated": v,
                  "pct_violated": round(100 * v / n, 3) if n else np.nan, "note": note})

# 1) Identities: the 8 rights should be a complete partition of a village's parcels
add_rule("sum(8 rights _jumlah) == jumlah_persil", "identity", desa[CNT].sum(axis=1) != desa["jumlah_persil"])
add_rule("sum(8 rights _luas) == luas_persil", "identity", desa[ARE].sum(axis=1) != desa["luas_persil"])

# 2) Zero coupling: a right has zero parcels exactly when it has zero area
for r in RIGHTS:
    add_rule(f"{r}: jumlah == 0  <=>  luas == 0", "identity", (desa[f"{r}_jumlah"] == 0) != (desa[f"{r}_luas"] == 0))
add_rule("jumlah_persil == 0  <=>  luas_persil == 0", "identity", (desa["jumlah_persil"] == 0) != (desa["luas_persil"] == 0))

# 3) Inequalities: one area can never exceed the area that contains it
TOL = 1.0001     # flood / RDTR areas are rounded to 0.1 m2, so allow 0.01% slack
fsum = desa[FLOOD].sum(axis=1)
add_rule("flood(low+mid+high) <= luas_desa", "inequality", fsum > desa["luas_desa"] * TOL)
add_rule("rdtr_luas <= luas_desa", "inequality", desa["rdtr_luas"] > desa["luas_desa"] * TOL)
add_rule("luas_persil <= luas_desa", "inequality (soft)", desa["luas_persil"] > desa["luas_desa"],
         scope=desa["luas_desa"] > 0, note="parcels can overlap / boundaries may not match; rows with luas_desa == 0 are skipped")
has_year = desa[YEARS].notna().all(axis=1)
add_rule("tahun_min <= tahun_median <= tahun_max", "inequality",
         (desa["tahun_min"] > desa["tahun_median"]) | (desa["tahun_median"] > desa["tahun_max"]), scope=has_year)

# 4) Null patterns: when are the year columns empty?
titled = desa["jumlah_persil"] - desa["kosong_jumlah"]       # parcels that carry some right (assumes kosong = no right)
yr_null = desa[YEARS].isna().all(axis=1)
add_rule("tahun_* are null together", "null pattern", desa[YEARS].isna().any(axis=1) & ~yr_null)
add_rule("year present  =>  at least 1 titled parcel", "implication", titled == 0, scope=has_year)
add_rule("no titled parcel  =>  year is null", "implication", ~yr_null, scope=(titled == 0))
add_rule("year is null  =>  no titled parcel", "implication", titled > 0, scope=yr_null)

rules = pd.DataFrame(RULES)
rules

In [ ]:
# --- The rows that break the "almost always true" rules ---------------------------------
# (a) year is null although titled parcels exist
display(desa.loc[yr_null & (titled > 0), ["kode8", "nama", "kabupaten", "jumlah_persil", "kosong_jumlah", "hak_milik_jumlah", "hgb_jumlah"]])

# (b) kosong has parcels but zero area
display(desa.loc[(desa["kosong_jumlah"] > 0) & (desa["kosong_luas"] == 0), ["kode8", "nama", "kabupaten", "jumlah_persil", "kosong_jumlah", "kosong_luas"]])

# (c) how the year columns relate to the titled count
pd.crosstab(yr_null.rename("year is null"), (titled == 0).rename("no titled parcel"))

In [ ]:
# --- Hierarchy: does the 8-digit code tell the same story as the names? ----------------
# Names are written inconsistently (KUTA ALAM vs Kuta Alam, double spaces), so also test a normalized version.
def norm(s):
    return s.str.upper().str.replace(r"\s+", " ", regex=True).str.strip()

for c in ["provinsi", "kabupaten", "kecamatan", "nama"]:
    desa[f"{c}_n"] = norm(desa[c])

def hierarchy_check(code_col, name_col, parent_col=None):
    """code -> name should be 1:1. Within the same parent, a name should map to a single code."""
    row = {"code": code_col, "name": name_col, "n_codes": desa[code_col].nunique(),
           "n_names_raw": desa[name_col].nunique(), "n_names_norm": desa[f"{name_col}_n"].nunique()}
    for tag, col in (("raw", name_col), ("norm", f"{name_col}_n")):
        row[f"codes_with_>1_name_{tag}"] = int((desa.groupby(code_col)[col].nunique() > 1).sum())
    keys = [parent_col, f"{name_col}_n"] if parent_col else [f"{name_col}_n"]
    row["names_with_>1_code_same_parent"] = int((desa.groupby(keys)[code_col].nunique() > 1).sum())
    return row

hier = pd.DataFrame([
    hierarchy_check("prov_code", "provinsi"),
    hierarchy_check("kab_code", "kabupaten", "prov_code"),
    hierarchy_check("kec_code", "kecamatan", "kab_code"),
])
display(hier)

# Province code vs province name: which rows carry a code that belongs to another province?
modal = desa.groupby("provinsi")["prov_code"].agg(lambda s: s.value_counts().idxmax())
desa["prov_code_mismatch"] = desa["prov_code"] != desa["provinsi"].map(modal)
print("rows where prov_code disagrees with provinsi:", int(desa["prov_code_mismatch"].sum()))
display(desa.loc[desa["prov_code_mismatch"], ["kode8", "prov_code", "nama", "kabupaten", "provinsi"]].assign(
    expected_code=lambda x: x["provinsi"].map(modal)))

# Kabupaten names that hide two different kab/kota codes inside the SAME province (Kabupaten X vs Kota X)
multi = desa.groupby(["prov_code", "kabupaten_n"])["kab_code"].agg(lambda s: sorted(s.unique()))
multi = multi[multi.str.len() > 1]
print("kabupaten names with more than one kab_code in the same province:", len(multi))
display(multi.head(12))

# ...and the opposite problem: one kab_code shared by two different kabupaten names
names_per_code = desa.groupby("kab_code")["kabupaten_n"].agg(lambda s: sorted(s.dropna().unique()))
shared = names_per_code[names_per_code.str.len() > 1].to_frame("kabupaten_names")
shared["n_villages"] = desa.groupby("kab_code").size()
print("kab_code shared by more than one kabupaten name:", len(shared))
shared

In [ ]:
# --- Same-name villages inside one kecamatan (name is not a key) -------------------------
dup = desa[desa.duplicated(["kec_code", "nama_n"], keep=False)].sort_values(["kec_code", "nama_n"])
print("rows involved:", len(dup), "| groups:", dup.groupby(["kec_code", "nama_n"]).ngroups)
dup[["kode8", "nama", "tipe", "kecamatan", "kabupaten", "luas_desa", "jumlah_persil"]].head(10)

**What Step 1 tells us (read it against the rule-book table above)**

- `jumlah_persil` and `luas_persil` are *derived* (sum of the 8 rights), so keep them out of any "which columns are related" analysis, or use them only as denominators.
- For every right, the count and area columns are zero together, so they carry the same presence/absence information. Treat each right as one concept with two measurements.
- `year present => titled parcel` holds with no exceptions, which supports reading `kosong` as "parcel with no registered right".
- Flood and RDTR areas never exceed the village area, so `banjir_*` and `rdtr_luas` can be turned into shares of `luas_desa`.
- `luas_persil <= luas_desa` is a *soft* rule and is violated often; that is a data-quality signal handled in Step 3.
- The 8-digit `kode` is a reliable hierarchy (province > kab/kota > kecamatan > village); only a handful of rows carry another province's code (flagged in `prov_code_mismatch`).
- Neither names nor codes alone identify a kabupaten. A name can hide a Kabupaten *and* a Kota (e.g. Bandung, Bekasi), and a few `kab_code`s are shared by two names (many pairs look like a regency and the one it was split from, e.g. Malaka/Belu; treat that as a hypothesis to confirm). Build a composite key (province + normalized name + code) before grouping at kabupaten level.
- Casing is inconsistent (UPPER vs Title), so always use the normalized `*_n` columns for names, never the raw ones.

## Step 2: Size-neutral features

**Goal:** convert raw counts and areas into shares, densities and sizes.

**Why:** almost every metric grows with village size, so raw correlations mostly say "big village means big everything". Ratios remove that effect and carry meaning of their own (e.g. average parcel size separates urban plots from plantation concessions).

Where a denominator is zero the feature is `NaN` (not `inf`, not 0): the ratio is undefined, and a fake 0 would create false patterns.

In [ ]:
# --- Helpers ----------------------------------------------------------------------------
FEATURE_GROUPS = {}     # group name -> list of feature columns (re-run safe)

def safe_div(a, b):
    """a / b, but NaN (not inf) where b == 0."""
    return a / b.where(b != 0)

def add_cols(feats, group=None):
    """Add or overwrite columns on `desa` in one go (re-run safe, avoids a fragmented frame)."""
    global desa
    desa = pd.concat([desa.drop(columns=[c for c in feats if c in desa.columns]),
                      pd.DataFrame(feats, index=desa.index)], axis=1)
    if group:
        FEATURE_GROUPS[group] = list(feats)

# --- 2a. Titled vs untitled land ----------------------------------------------------------
add_cols({
    "titled_jumlah": desa["jumlah_persil"] - desa["kosong_jumlah"],
    "titled_luas": desa["luas_persil"] - desa["kosong_luas"],
}, group=None)
add_cols({
    "titled_share_jumlah": safe_div(desa["titled_jumlah"], desa["jumlah_persil"]),
    "titled_share_luas": safe_div(desa["titled_luas"], desa["luas_persil"]),
}, group="titled")

# --- 2b. Composition: share of parcels, share of area, mean parcel size per right ----------
share_cnt = {f"{r}_share_jumlah": safe_div(desa[f"{r}_jumlah"], desa["jumlah_persil"]) for r in RIGHTS}
share_are = {f"{r}_share_luas": safe_div(desa[f"{r}_luas"], desa["luas_persil"]) for r in RIGHTS}
mean_size = {f"{r}_mean_luas": safe_div(desa[f"{r}_luas"], desa[f"{r}_jumlah"]) for r in RIGHTS}   # m2 per parcel
add_cols(share_cnt, group="share_jumlah")
add_cols(share_are, group="share_luas")
add_cols(mean_size, group="mean_size")

# --- 2c. Density and coverage -------------------------------------------------------------
add_cols({
    "parcel_density": safe_div(desa["jumlah_persil"], desa["luas_desa"] / 1e4),      # parcels per hectare
    "mean_parcel_m2": safe_div(desa["luas_persil"], desa["jumlah_persil"]),            # average parcel size
    "coverage": safe_div(desa["luas_persil"], desa["luas_desa"]),                      # parcel area / village area
}, group="density")

# --- 2d. Hazard and spatial plan -------------------------------------------------------------
fsum = desa[FLOOD].sum(axis=1)
add_cols({
    "flood_low_share": safe_div(desa["banjir_rendah"], desa["luas_desa"]),
    "flood_mid_share": safe_div(desa["banjir_sedang"], desa["luas_desa"]),
    "flood_high_share": safe_div(desa["banjir_tinggi"], desa["luas_desa"]),
    "flood_any_share": safe_div(fsum, desa["luas_desa"]).clip(upper=1),        # clip: tiny rounding overshoot
    "flood_high_of_flooded": safe_div(desa["banjir_tinggi"], fsum),            # severity among flooded land
    "rdtr_share": safe_div(desa["rdtr_luas"], desa["luas_desa"]).clip(upper=1),
}, group="hazard_plan")

# --- 2e. Registration years ---------------------------------------------------------------------
REF_YEAR = int(desa["dihitung_pada"].dt.year.max())     # year the snapshot was computed
add_cols({
    "year_span": desa["tahun_max"] - desa["tahun_min"],      # years between first and last registration
    "age_first": REF_YEAR - desa["tahun_min"],
    "age_median": REF_YEAR - desa["tahun_median"],
    "age_last": REF_YEAR - desa["tahun_max"],                # 0 = still being registered this year
}, group="years")

print("reference year:", REF_YEAR)
print({g: len(c) for g, c in FEATURE_GROUPS.items()}, "| total features:", sum(len(c) for c in FEATURE_GROUPS.values()))

In [ ]:
# --- Feature sanity table: how many values are defined, and what do they look like? ---------
ALL_FEATS = [c for cols in FEATURE_GROUPS.values() for c in cols]
group_of = {c: g for g, cols in FEATURE_GROUPS.items() for c in cols}

feat_summary = desa[ALL_FEATS].describe(percentiles=[0.5]).T[["count", "min", "50%", "mean", "max"]]
feat_summary.insert(0, "group", [group_of[c] for c in feat_summary.index])
feat_summary["pct_undefined"] = (100 * (1 - feat_summary["count"] / len(desa))).round(1)
feat_summary["n_inf"] = np.isinf(desa[ALL_FEATS]).sum()
feat_summary.round(3)

**Reading the table:** `pct_undefined` is high for the per-right `*_mean_luas` and `*_share_*` columns of rare rights; that is expected (no parcels of that right means no mean size). `n_inf` must be 0 everywhere. The shares of the 8 rights in each group sum to 1, so they are *compositional*; this matters in Step 5.

## Step 3: Robust preparation

**Goal:** make the data safe for correlation work: flag bad rows, handle skew, and treat zero-heavy columns properly.

**Why:** a single 853-billion m² row dominates Pearson correlation, the area columns are extremely right-skewed, and several columns are 94-100% zeros. Without this step every later number is distorted.

Two parameters you can change and re-run: `RATIO_CAP` and `ZERO_THRESH`.

In [ ]:
# --- 3a. Flag problem rows (flag first, decide later) ------------------------------------------
RATIO_CAP = 10       # luas_persil / luas_desa above this = almost certainly a geometry / unit error

add_cols({
    # placeholder records that are not real villages (nama "HGU", or "-" which your SENTINELS turned into NaN)
    "is_pseudo": desa["nama"].isna() | desa["nama"].fillna("").str.strip().str.upper().eq("HGU"),
    "is_zero_area": desa["luas_desa"] == 0,
    "is_ratio_outlier": desa["coverage"] > RATIO_CAP,
    "is_overshoot": (desa["coverage"] > 1) & (desa["coverage"] <= RATIO_CAP),     # informational only
    "is_zero_parcels": desa["jumlah_persil"] == 0,                                 # informational only
})
desa["analysis_ok"] = ~(desa["is_pseudo"] | desa["is_zero_area"] | desa["is_ratio_outlier"])
ok = desa["analysis_ok"]

flags = ["is_pseudo", "is_zero_area", "is_ratio_outlier", "is_overshoot", "is_zero_parcels", "prov_code_mismatch"]
flag_table = pd.DataFrame({
    "n_rows": desa[flags].sum(),
    "pct": (100 * desa[flags].mean()).round(2),
    "excluded_from_analysis": [True, True, True, False, False, False],
})
display(flag_table)
print("analysis rows kept:", int(ok.sum()), "of", len(desa), f"({100 * ok.mean():.2f}%)")

# The excluded rows with the biggest parcel area
desa.loc[~ok, ["kode8", "nama", "kabupaten", "provinsi", "luas_desa", "luas_persil", "coverage", "is_pseudo", "is_zero_area"]
        ].sort_values("luas_persil", ascending=False).head(10)

In [ ]:
# --- 3b. Why RATIO_CAP = 10? Sensitivity of the cut-off -----------------------------------------
# Look for a plateau: once the cap is high enough, the aggregate ratio of the kept rows stops moving.
# Below ~2 you start throwing away villages whose parcels merely overlap or whose boundary is a bit off.
rows = []
for cap in [1, 1.5, 2, 5, 10, 50, 100]:
    out = desa["coverage"] > cap
    kept = ~out & ~desa["is_pseudo"] & ~desa["is_zero_area"]
    rows.append({
        "cap": cap,
        "rows_above_cap": int(out.sum()),
        "pct_rows": round(100 * out.mean(), 2),
        "share_of_all_luas_persil_%": round(100 * desa.loc[out, "luas_persil"].sum() / desa["luas_persil"].sum(), 1),
        "sum(luas_persil)/sum(luas_desa) of kept rows": round(desa.loc[kept, "luas_persil"].sum() / desa.loc[kept, "luas_desa"].sum(), 3),
    })
print("sum(luas_persil)/sum(luas_desa), all rows:", round(desa["luas_persil"].sum() / desa["luas_desa"].sum(), 3))
pd.DataFrame(rows)

In [ ]:
# --- 3c. Zero-heavy columns: split "is it present?" from "how big is it when present?" -----------
ZERO_THRESH = 0.5      # a metric with more than 50% zeros gets a presence flag

METRICS = ["luas_desa", "jumlah_persil", "luas_persil"] + CNT + ARE + ["rdtr_luas"] + FLOOD
zero_pct = (desa.loc[ok, METRICS] == 0).mean()
ZERO_INFLATED = zero_pct[zero_pct > ZERO_THRESH].index.tolist()

def base_name(c):
    for suffix in ("_jumlah", "_luas"):
        if c.endswith(suffix):
            return c[: -len(suffix)]
    return c

# one presence flag per concept (a right's count and area are zero together, see Step 1)
presence = {f"has_{base_name(c)}": desa[c] > 0 for c in ZERO_INFLATED}
add_cols(presence, group="presence")

zi = pd.DataFrame({
    "zero_%": (100 * zero_pct[ZERO_INFLATED]).round(1),
    "present_%": (100 * (desa.loc[ok, ZERO_INFLATED] > 0).mean()).round(1),
    "n_present": (desa.loc[ok, ZERO_INFLATED] > 0).sum(),
    "median_when_present": [desa.loc[ok & (desa[c] > 0), c].median() for c in ZERO_INFLATED],
    "p99_when_present": [desa.loc[ok & (desa[c] > 0), c].quantile(0.99) for c in ZERO_INFLATED],
})
print("zero-inflated metrics (> %d%% zeros):" % int(ZERO_THRESH * 100), len(ZERO_INFLATED))
zi.sort_values("zero_%", ascending=False)

In [ ]:
# --- 3d. Log transform: skewness before and after ----------------------------------------------------
# log10 of the positive values only; zeros become NaN because "zero" is handled by the has_* flags
LOG_COLS = METRICS + ["parcel_density", "mean_parcel_m2"]
add_cols({f"{c}_log10": np.log10(desa[c].where(desa[c] > 0)) for c in LOG_COLS}, group="log10")

skew = pd.DataFrame({
    "zero_%": [round(100 * (desa.loc[ok, c] == 0).mean(), 1) for c in LOG_COLS],
    "skew_raw": [desa.loc[ok, c].skew() for c in LOG_COLS],
    "skew_log10": [desa.loc[ok, f"{c}_log10"].skew() for c in LOG_COLS],
}, index=LOG_COLS)
skew.sort_values("skew_raw", ascending=False).round(2)

In [ ]:
# --- 3e. Why Spearman / log instead of plain Pearson? -------------------------------------------------
pairs = [("luas_desa", "luas_persil"), ("luas_desa", "jumlah_persil"), ("luas_desa", "hak_milik_luas"),
         ("luas_desa", "banjir_tinggi"), ("luas_desa", "rdtr_luas"), ("hak_milik_jumlah", "hak_milik_luas")]

rows = []
for a, b in pairs:
    both = ok & (desa[a] > 0) & (desa[b] > 0)
    rows.append({
        "pair": f"{a} ~ {b}",
        "pearson_raw_all_rows": desa[a].corr(desa[b]),
        "pearson_raw_clean_rows": desa.loc[ok, a].corr(desa.loc[ok, b]),
        "pearson_log10_clean": desa.loc[both, f"{a}_log10"].corr(desa.loc[both, f"{b}_log10"]),
        "spearman_clean": desa.loc[both, a].corr(desa.loc[both, b], method="spearman"),
        "n_pairs": int(both.sum()),
    })
pd.DataFrame(rows).set_index("pair").round(3)

In [ ]:
# --- 3f. See it: raw vs log, and where the cut-off falls ---------------------------------------------------
fig, ax = plt.subplots(2, 2, figsize=(12, 8))

ax[0, 0].hist(desa["luas_persil"], bins=60, color="tab:red")
ax[0, 0].set_title("luas_persil, raw, all rows: one outlier flattens everything")
ax[0, 0].set_xlabel("m2")

ax[0, 1].hist(desa.loc[ok, "luas_persil_log10"].dropna(), bins=60, color="tab:blue")
ax[0, 1].set_title("log10(luas_persil), clean rows: readable shape")
ax[0, 1].set_xlabel("log10 m2")

cov = np.log10(desa["coverage"].where(desa["coverage"] > 0)).dropna()
ax[1, 0].hist(cov, bins=80, color="tab:gray")
ax[1, 0].axvline(0, color="k", ls="--", lw=1, label="parcel area = village area")
ax[1, 0].axvline(np.log10(RATIO_CAP), color="tab:red", ls="--", lw=1, label=f"RATIO_CAP = {RATIO_CAP}")
ax[1, 0].set_yscale("log")
ax[1, 0].set_title("coverage = luas_persil / luas_desa (log10 axis)")
ax[1, 0].legend()

pos = (desa["luas_desa"] > 0) & (desa["luas_persil"] > 0)
ax[1, 1].scatter(desa.loc[pos & ok, "luas_desa"], desa.loc[pos & ok, "luas_persil"], s=2, alpha=0.2, label="kept")
ax[1, 1].scatter(desa.loc[pos & ~ok, "luas_desa"], desa.loc[pos & ~ok, "luas_persil"], s=14, color="tab:red", label="excluded")
lim = [desa.loc[pos, "luas_desa"].min(), desa.loc[pos, "luas_desa"].max()]
ax[1, 1].plot(lim, lim, "k--", lw=1, label="y = x")
ax[1, 1].set_xscale("log"); ax[1, 1].set_yscale("log")
ax[1, 1].set_xlabel("luas_desa (m2)"); ax[1, 1].set_ylabel("luas_persil (m2)")
ax[1, 1].set_title("parcel area vs village area"); ax[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- 3g. The analysis frame for the next steps -----------------------------------------------------------------
desaOK = desa.loc[desa["analysis_ok"]].copy()

print("desa   (all rows + flags + features):", desa.shape)
print("desaOK (analysis rows only)          :", desaOK.shape)
print()
for g, cols in FEATURE_GROUPS.items():
    print(f"{g:<14} {len(cols):>2} cols  e.g. {cols[:3]}")

# Optional: keep the cleaned frame for later sessions
# desaOK.to_csv(DATA / "desa_clean.csv", index=False)

**Handing over to Step 4.** From here on use `desaOK` (flagged rows removed) and the feature groups in `FEATURE_GROUPS`. Use the `*_log10` columns and the `has_*` flags for zero-heavy metrics, Spearman instead of Pearson for skewed pairs, and remember that the shares of the 8 rights are compositional (Step 5).

## Step 3b: Anomaly audit

Step 3 *flagged and excluded* the anomalous rows. This section *investigates* them. Run Steps 1-3 first; every cell below builds on `desa` and its flags.

**Goal:** answer four questions about the anomalies we found:
1. Are they one systematic problem or several separate ones? (do the flags overlap?)
2. Are they concentrated in certain regions? (a data-pipeline or coverage issue) or scattered? (random noise)
3. For the ambiguous ones (flood zeros, area outliers), which interpretation does the data support?
4. Do our treatment choices change the conclusions? (sensitivity)

It also finishes three loose ends from Step 3: a kabupaten key, a group for zero-parcel villages, and a side table of the excluded rows.

In [ ]:
# --- 3b-1. One anomaly matrix: every flag, how big, and what we do with it ------------------------
ok = desa["analysis_ok"]
fsum = desa[FLOOD].sum(axis=1)
titled = desa["jumlah_persil"] - desa["kosong_jumlah"]

add_cols({
    # no flood area in any class: a true "no hazard", or "not computed"? (investigated below)
    "flood_all_zero": fsum == 0,
    # the 12 tiny rule breaks from Step 1 (kosong count>0 but area 0; year null although titled parcels exist)
    "is_minor_rule_break": ((desa["kosong_jumlah"] > 0) & (desa["kosong_luas"] == 0))
                           | (desa[YEARS].isna().all(axis=1) & (titled > 0)),
})

ANOMALY_FLAGS = ["is_pseudo", "is_zero_area", "is_ratio_outlier", "is_overshoot",
                 "is_zero_parcels", "prov_code_mismatch", "flood_all_zero", "is_minor_rule_break"]
TREATMENT = {
    "is_pseudo": "excluded (not a village)",
    "is_zero_area": "excluded (ratios undefined)",
    "is_ratio_outlier": "excluded (almost certainly an error)",
    "is_overshoot": "kept; use coverage_log10 / coverage_clip1",
    "is_zero_parcels": "kept as its own group (parcel_status)",
    "prov_code_mismatch": "kept (label problem only)",
    "flood_all_zero": "kept; do not use flood features for clustering until confirmed",
    "is_minor_rule_break": "kept (immaterial)",
}
anomalies = pd.DataFrame({
    "n_rows": desa[ANOMALY_FLAGS].sum(),
    "pct_rows": (100 * desa[ANOMALY_FLAGS].mean()).round(2),
    "n_excluded": [int((desa[f] & ~ok).sum()) for f in ANOMALY_FLAGS],
    "treatment": [TREATMENT[f] for f in ANOMALY_FLAGS],
})
anomalies

In [ ]:
# --- 3b-2. Do the anomalies overlap? One systematic problem or several separate ones? ---------------
A = desa[ANOMALY_FLAGS].astype(int)
overlap = A.T @ A       # diagonal = rows per flag, off-diagonal = rows carrying both flags
display(overlap)

n_flags = A.sum(axis=1)
print("number of anomaly flags per row (rows):", n_flags.value_counts().sort_index().to_dict())

# Is each overlap more than coincidence? lift = observed / expected if the two flags were independent
#   lift ~ 1  -> coincidence      lift >> 1 -> shared cause (or one flag implies the other by construction)
N = len(desa)
n_each = pd.Series(np.diag(overlap), index=ANOMALY_FLAGS)
rows = []
for i, a in enumerate(ANOMALY_FLAGS):
    for b in ANOMALY_FLAGS[i + 1:]:
        observed = int(overlap.loc[a, b])
        if observed:
            expected = n_each[a] * n_each[b] / N
            rows.append({"flag_a": a, "flag_b": b, "rows_with_both": observed,
                         "expected_if_independent": round(expected, 1), "lift": round(observed / expected, 2)})
pd.DataFrame(rows).sort_values("rows_with_both", ascending=False).reset_index(drop=True)

In [ ]:
# --- 3b-3. Are the anomalies concentrated in certain regions? -----------------------------------------
def cramers_v(a, b):
    """Bias-corrected Cramer's V between two categorical series: 0 = unrelated, 1 = fully determined by the other."""
    ct = pd.crosstab(a, b).to_numpy().astype(float)
    n = ct.sum()
    r, k = ct.shape
    if min(r, k) < 2:
        return np.nan
    expected = ct.sum(axis=1, keepdims=True) @ ct.sum(axis=0, keepdims=True) / n
    phi2 = ((ct - expected) ** 2 / expected).sum() / n
    phi2_corr = max(0.0, phi2 - (k - 1) * (r - 1) / (n - 1))
    r_corr, k_corr = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return float(np.sqrt(phi2_corr / min(k_corr - 1, r_corr - 1)))

MIN_N = 30      # below this a concentration statistic is noise, so only list the rows
rows = []
for f in ANOMALY_FLAGS:
    by_prov = desa.groupby("provinsi")[f].sum().sort_values(ascending=False)
    n = int(by_prov.sum())
    rows.append({
        "flag": f, "n_rows": n,
        "n_provinces": int((by_prov > 0).sum()),
        "top_province": by_prov.index[0],
        "top_1_share_%": round(100 * by_prov.iloc[0] / n, 1),
        "top_3_share_%": round(100 * by_prov.iloc[:3].sum() / n, 1),
        "V_province": round(cramers_v(desa["provinsi"], desa[f]), 3) if n >= MIN_N else np.nan,
        "V_kabupaten": round(cramers_v(desa["kab_code"], desa[f]), 3) if n >= MIN_N else np.nan,
    })
conc = pd.DataFrame(rows).set_index("flag")
print("V near 0 = scattered everywhere (noise); V near 1 = fully determined by location (systematic).")
print("Compare flags with each other. V_kabupaten > V_province means the pattern is local, not just provincial.")
conc

In [ ]:
# --- 3b-4. Flood zeros: "no hazard" or "not computed"? ------------------------------------------------
# Evidence 1: how the zero rate varies by province
flood_prov = desa.groupby("provinsi")["flood_all_zero"].agg(n_villages="size", n_zero="sum", zero_rate="mean").sort_values("zero_rate", ascending=False)
flood_prov["zero_rate"] = flood_prov["zero_rate"].round(3)
display(pd.concat([flood_prov.head(6), flood_prov.tail(4)]))

# Evidence 2: is it scattered randomly inside each kabupaten, or do whole kabupaten sit at 0% / 100%?
p = desa["flood_all_zero"].mean()
kab = desa.groupby(["provinsi_n", "kabupaten_n", "kab_code"])["flood_all_zero"].agg(n_villages="size", n_zero="sum").reset_index()
kab = kab[kab["n_villages"] >= 5].copy()
kab["zero_rate"] = kab["n_zero"] / kab["n_villages"]
obs_all, obs_none = int((kab["n_zero"] == kab["n_villages"]).sum()), int((kab["n_zero"] == 0).sum())
exp_all, exp_none = float((p ** kab["n_villages"]).sum()), float(((1 - p) ** kab["n_villages"]).sum())
print(f"national zero rate: {p:.3f} | kabupaten groups with >= 5 villages: {len(kab)}")
print(f"kabupaten where EVERY village is zero : observed {obs_all:>3} | expected if zeros were scattered at random {exp_all:.1f}")
print(f"kabupaten where NO village is zero    : observed {obs_none:>3} | expected if zeros were scattered at random {exp_none:.1f}")

def share_explained(group):
    """Share of the variance of the 0/1 flag that is explained by the group means (eta squared)."""
    y = desa["flood_all_zero"].astype(float)
    return float(((y.groupby(desa[group]).transform("mean") - y.mean()) ** 2).sum() / ((y - y.mean()) ** 2).sum())
print(f"variance explained by province: {share_explained('provinsi'):.3f} | by kabupaten: {share_explained('kab_code'):.3f}")

# Evidence 3: is it just tiny villages that a hazard map misses?
c = desa[ok].copy()
c["size_decile"] = pd.qcut(c["luas_desa"], 10, labels=False) + 1
display(c.groupby("size_decile").agg(median_luas_desa=("luas_desa", "median"), zero_rate=("flood_all_zero", "mean")).round(3).T)

# The kabupaten that look most like "not computed": (nearly) every village at zero
flood_suspect_kab = kab[kab["zero_rate"] >= 0.9].sort_values(["zero_rate", "n_villages"], ascending=False)
print("kabupaten with >= 90% of villages at zero flood area:", len(flood_suspect_kab))
flood_suspect_kab[["provinsi_n", "kabupaten_n", "kab_code", "n_villages", "n_zero", "zero_rate"]].round(3)

In [ ]:
# --- 3b-5. Ratio outliers: which side of the ratio is wrong, and what are these rows? ------------------
out = desa["is_ratio_outlier"]
print("ratio outliers:", int(out.sum()))
print("top provinces:", desa.loc[out, "provinsi"].value_counts().head(6).to_dict())
print("top-1 province share: %.0f%%  (low = scattered, not one region)" % (100 * desa.loc[out, "provinsi"].value_counts(normalize=True).iloc[0]))

# Which land class carries the huge area?
ks = desa.loc[out, "kosong_share_luas"]
print(f"kosong share of luas_persil among outliers: median {ks.median():.2f} | rows with kosong >= 90% of area: {int((ks >= 0.9).sum())} of {int(out.sum())}")

# Which side looks abnormal? Compare the village area with the typical village in the same kabupaten.
kab_median_area = desa.loc[ok].groupby("kab_code")["luas_desa"].median()
add_cols({"desa_vs_kab_median": desa["luas_desa"] / desa["kab_code"].map(kab_median_area)})
village_typical = desa["desa_vs_kab_median"].between(0.1, 10)      # village area within 10x of its kabupaten's median
add_cols({"outlier_culprit": np.where(~out, "", np.where(village_typical, "parcel area (village looks normal)", "village area or both"))})
print()
print("heuristic: village area typical for its kabupaten  =>  the inflated side is most likely the parcel area")
print(desa.loc[out, "outlier_culprit"].value_counts().to_string())

# Contrast with the overshoot rows we KEEP: do they look like normal villages with mild overlap?
nat_parcel = desa.loc[ok, "mean_parcel_m2"].median()
contrast = pd.DataFrame({
    "n_rows": [int(out.sum()), int(desa["is_overshoot"].sum()), int(ok.sum())],
    "median_coverage": [desa.loc[m, "coverage"].median() for m in (out, desa["is_overshoot"], ok)],
    "median_mean_parcel_vs_national": [(desa.loc[m, "mean_parcel_m2"] / nat_parcel).median() for m in (out, desa["is_overshoot"], ok)],
    "median_village_vs_kab_median": [desa.loc[m, "desa_vs_kab_median"].median() for m in (out, desa["is_overshoot"], ok)],
}, index=["ratio outliers (excluded)", "overshoot 1-10x (kept)", "all analysis rows"]).round(2)
contrast

In [ ]:
# --- 3b-6. Zero-parcel villages become their own group ---------------------------------------------------
titled = desa["jumlah_persil"] - desa["kosong_jumlah"]
add_cols({"parcel_status": np.select([desa["jumlah_persil"] == 0, titled == 0], ["no_parcels", "untitled_only"], default="has_titled")})

g = desa[ok].groupby("parcel_status")
profile_status = pd.DataFrame({
    "n_villages": g.size(),
    "pct_of_analysis_rows": (100 * g.size() / ok.sum()).round(2),
    "pct_kelurahan": (100 * g["tipe"].apply(lambda s: (s == "Kelurahan").mean())).round(1),
    "median_luas_desa_m2": g["luas_desa"].median().round(0),
    "pct_with_rdtr": (100 * g["rdtr_luas"].apply(lambda s: (s > 0).mean())).round(1),
    "pct_flood_all_zero": (100 * g["flood_all_zero"].mean()).round(1),
})
display(profile_status)
print("where the villages without any parcel are:", desa.loc[ok & (desa["parcel_status"] == "no_parcels"), "provinsi"].value_counts().head(6).to_dict())
print("villages with parcels but none titled (untitled_only), all rows:", int((desa["parcel_status"] == "untitled_only").sum()))

In [ ]:
# --- 3b-7. A kabupaten key that survives Kab/Kota twins and stray legacy codes -----------------------------
# Name alone merges Kabupaten X with Kota X; code alone is sometimes shared by a regency and the one it split from.
# Key = province + normalized name + code, except that "stray" codes (a handful of villages that still carry a legacy
# code) are re-assigned to the dominant code of the same province + name.
STRAY_MAX = 5

keys3 = ["provinsi_n", "kabupaten_n", "kab_code"]
g = desa.groupby(keys3).size().rename("n_villages").reset_index()
dominant = (g.sort_values("n_villages", ascending=False)
             .drop_duplicates(["provinsi_n", "kabupaten_n"])
             .set_index(["provinsi_n", "kabupaten_n"])["kab_code"].rename("dominant_code"))
g = g.join(dominant, on=["provinsi_n", "kabupaten_n"])
g["kab_code_main"] = g["kab_code"].where(g["n_villages"] >= STRAY_MAX, g["dominant_code"])

lookup = g.set_index(keys3)["kab_code_main"]
add_cols({"kab_code_main": lookup.reindex(pd.MultiIndex.from_frame(desa[keys3])).to_numpy()})
add_cols({"kab_key": desa["provinsi_n"] + " | " + desa["kabupaten_n"] + " | " + desa["kab_code_main"]})

print("keys by (province, name, code):", len(g), "| after re-assigning stray codes:", desa["kab_key"].nunique())
print("each kab_key sits in exactly one province:", bool((desa.groupby("kab_key")["provinsi_n"].nunique() == 1).all()))
display(g.loc[g["kab_code"] != g["kab_code_main"], ["provinsi_n", "kabupaten_n", "kab_code", "n_villages", "kab_code_main"]])

# names that still map to more than one key: these should be real Kabupaten/Kota twins (or one regency split over two code blocks)
twins = desa.groupby(["provinsi_n", "kabupaten_n"])["kab_key"].nunique()
print("names that still have >1 key (Kab/Kota twins, splits):", int((twins > 1).sum()))

In [ ]:
# --- 3b-8. Coverage features for the rows we keep + a side table for the rows we exclude ---------------------
add_cols({
    "coverage_log10": np.log10(desa["coverage"].where(desa["coverage"] > 0)),   # symmetric around 1 = "parcels exactly fill the village"
    "coverage_clip1": desa["coverage"].clip(upper=1),                           # saturates at 1; is_overshoot keeps the information
}, group="coverage_variants")

reason_labels = {"is_pseudo": "pseudo-record", "is_zero_area": "zero village area", "is_ratio_outlier": f"parcel/village area > {RATIO_CAP}x"}
excluded = desa.loc[~ok, ["kode8", "nama", "kecamatan", "kabupaten", "provinsi", "luas_desa", "jumlah_persil",
                          "luas_persil", "coverage", "kosong_share_luas", "outlier_culprit"]].copy()
excluded["reason"] = desa.loc[~ok, list(reason_labels)].apply(
    lambda r: "; ".join(label for flag, label in reason_labels.items() if r[flag]), axis=1)
excluded = excluded.sort_values("luas_persil", ascending=False)

print("excluded rows:", len(excluded))
display(excluded["reason"].value_counts())
# excluded.to_csv(DATA / "desa_excluded_rows.csv", index=False)      # optional: keep the audit trail
excluded.head(12)

In [ ]:
# --- 3b-9. Sensitivity: do our treatment choices change the conclusions? ----------------------------------------
SCENARIOS = {
    "no exclusions at all": pd.Series(True, index=desa.index),
    "base (analysis_ok)": ok,
    "base + drop overshoot rows": ok & ~desa["is_overshoot"],
    "base + drop zero-parcel villages": ok & ~desa["is_zero_parcels"],
    "base + drop province-code mismatch": ok & ~desa["prov_code_mismatch"],
    "base + drop flood-zero villages": ok & ~desa["flood_all_zero"],
}

def scenario_stats(mask):
    d = desa[mask]
    return {
        "n_rows": len(d),
        "median titled_share_luas": d["titled_share_luas"].median(),
        "median hak_milik_share_luas": d["hak_milik_share_luas"].median(),
        "median parcel_density": d["parcel_density"].median(),
        "median mean_parcel_m2": d["mean_parcel_m2"].median(),
        "spearman luas_desa ~ jumlah_persil": d["luas_desa"].corr(d["jumlah_persil"], method="spearman"),
        "spearman luas_desa ~ luas_persil": d["luas_desa"].corr(d["luas_persil"], method="spearman"),
        "pearson luas_desa ~ luas_persil": d["luas_desa"].corr(d["luas_persil"]),
        "mean luas_persil (m2)": d["luas_persil"].mean(),
        "kosong share of total area": d["kosong_luas"].sum() / d["luas_persil"].sum(),
    }

sens = pd.DataFrame({name: scenario_stats(m) for name, m in SCENARIOS.items()})
delta = (100 * (sens.sub(sens["base (analysis_ok)"], axis=0)).div(sens["base (analysis_ok)"].abs(), axis=0)).drop(index="n_rows").round(1)

def fmt(v):
    return f"{v:,.0f}" if abs(v) >= 1000 else f"{v:.3f}"

print("statistics per scenario:")
display(sens.apply(lambda col: col.map(fmt)))
print("percent change versus the base scenario (small = the choice does not matter for that statistic):")
delta

In [ ]:
# --- 3b-10. Refresh the analysis frame so it carries the new columns -----------------------------------------------
desaOK = desa.loc[desa["analysis_ok"]].copy()

print("desa   :", desa.shape, "| desaOK:", desaOK.shape)
print("new columns this section:", ["flood_all_zero", "is_minor_rule_break", "desa_vs_kab_median", "outlier_culprit",
                                    "parcel_status", "kab_code_main", "kab_key", "coverage_log10", "coverage_clip1"])
print("feature groups:", {g: len(c) for g, c in FEATURE_GROUPS.items()})

**How to read the audit**

- *Overlap and lift (3b-2):* if two flags co-occur about as often as chance predicts (lift near 1), they are separate issues and can be handled separately. A lift well above 1 means a shared cause, unless one flag implies the other by construction (every zero-area village also has zero parcels).
- *Concentration table (3b-3):* a flag with `V` close to 0 is scattered (random noise); a high `V_kabupaten` means the pattern follows administrative units, which usually points to how the source data was produced rather than to real-world variation.
- *Flood zeros (3b-4):* three kinds of evidence. A zero rate that follows geography (hilly vs low-lying provinces) supports "real zero". Whole kabupaten at 0% or 100% (more than random scatter predicts) supports a regional coverage effect. A zero rate that falls with village size would suggest a map-resolution artefact. This data cannot settle the question alone; `flood_suspect_kab` lists the kabupaten to ask the data owner about.
- *Ratio outliers (3b-5):* the culprit column is a heuristic, not proof. Its purpose is to tell whoever owns the source which field to check.
- *Sensitivity (3b-9):* statistics whose percent change is near 0 across scenarios (medians, rank correlations) do not depend on our exclusion choices. Statistics that move a lot (means, sums, Pearson) do, which is exactly why Step 3 excludes the extreme rows. For statistics that sit near zero (such as the Spearman correlation of `luas_desa` and `jumlah_persil`) a percent change is misleading; read the absolute values in the first table instead.

**For the clustering stage:** fit on `desaOK`, then assign the excluded rows (`excluded`) to their nearest cluster afterwards, so they stay visible without distorting the cluster centres. Use `parcel_status` to keep zero-parcel villages out of share-based distances, and `kab_key` (not names or raw codes) for any kabupaten-level grouping.

## Step 4: Clustering-ready data

**Goal:** turn `desa` into matrices a modeller can feed straight into a clustering algorithm, with every decision behind them written down. Modelling itself is out of scope here.

**Design**
- **Two variants.** **A** = land-tenure profile only (default). **B** = A plus two vetted extras (village size, registration age).
- **Unscaled and scaled** matrices for each variant, because some methods scale internally.
- **Villages that cannot be fitted** (no parcels, or excluded in Step 3) are not dropped silently: they go to a `place_later` table with the reason.
- **Interpretation-only columns** (flood, RDTR, years, geography) are kept in a separate table, so clusters can be profiled without those columns having shaped them.
- **A decision log** (options, default, reason, measured evidence) and a **recipe** (every parameter needed to re-apply the transform).

Run Steps 1-3b first. Nothing below needs a package beyond numpy, pandas and matplotlib.

In [ ]:
# --- 4-1. Parameters and column roles ------------------------------------------------------------
RARE_AREA_SHARE = 0.01        # a right holding < 1% of all parcel area is grouped into "other"
SAMPLE_N = 1500               # villages used for the distance-comparison evidence (4-7)
SEED = 42
EXPORT = True                 # write the deliverables to disk in 4-10
EXPORT_DIR = DATA / "clustering_ready"

ok = desa["analysis_ok"]
SHARE_AREA = [f"{r}_share_luas" for r in RIGHTS]
SHARE_COUNT = [f"{r}_share_jumlah" for r in RIGHTS]

roles = []
def role(cols, name, why):
    roles.extend({"column": c, "role": name, "why": why} for c in cols)

role(SHARE_AREA, "input (grouped + sqrt)", "area share of each right; rare rights are grouped in 4-3")
role(["mean_parcel_m2_log10", "parcel_density_log10"], "input", "two independent directions of the size / density / coverage identity (4-2)")
role(["luas_desa_log10", "age_median"], "input (variant B only)", "vetted extras (age enters as log1p): clean and defined for every titled village")
role(["coverage", "coverage_log10", "coverage_clip1"], "derived, not an input", "exact function of parcel density and mean parcel size (4-2)")
role(["titled_share_luas"], "derived, not an input", "equals 1 - kosong share (4-2)")
role(SHARE_COUNT + ["titled_share_jumlah"] + [f"{r}_mean_luas" for r in RIGHTS], "not used", "parcel-count shares and per-right sizes: see decision D3")
role(["flood_low_share", "flood_mid_share", "flood_high_share", "flood_any_share", "flood_high_of_flooded", "flood_all_zero"],
     "interpretation only", "zero flood area is ambiguous ('no hazard' vs 'not computed'), strongly regional")
role(["rdtr_share"], "interpretation only", "plan coverage exists for only ~17% of villages; zero is regional")
role(["tahun_min", "tahun_median", "tahun_max", "year_span", "age_first", "age_last"], "interpretation only", "undefined where no parcel is titled")
role(["provinsi", "kab_key", "tipe"], "interpretation only", "geography / administrative type: use to profile clusters, not to build them")
roles_df = pd.DataFrame(roles)
roles_df.groupby("role").agg(n_columns=("column", "size"), why=("why", "first"))

In [ ]:
# --- 4-2. Redundancy: which candidate columns are exact functions of others? ---------------------------
core = ok & (desa["parcel_status"] != "no_parcels")          # villages with at least one parcel
x = desa.loc[core]

# (a) titled share is just 1 - kosong share
dev_titled = (x["titled_share_luas"] + x["kosong_share_luas"] - 1).abs().max()
# (b) log10(density) + log10(mean parcel size) = log10(coverage) + 4   (parcels/ha * m2/parcel = m2 per ha = coverage * 10,000)
dev_log = (x["parcel_density_log10"] + x["mean_parcel_m2_log10"] - (x["coverage_log10"] + 4)).abs().max()
# (c) the 8 area shares sum to 1
dev_sum = (x[SHARE_AREA].sum(axis=1) - 1).abs().max()
print(f"max |titled_share + kosong_share - 1|                         : {dev_titled:.2e}")
print(f"max |log10 density + log10 mean parcel - (log10 coverage + 4)| : {dev_log:.2e}")
print(f"max |sum of 8 area shares - 1|                                : {dev_sum:.2e}")

# (d) linear dependence: the 3 log columns have rank 2, not 3
z = x[["parcel_density_log10", "mean_parcel_m2_log10", "coverage_log10"]]
z = (z - z.mean()) / z.std()          # mean-centred: the identity is a linear dependence only after centring
print("singular values of the 3 log columns (smallest ~ 0 = exact dependence):", np.linalg.svd(z.to_numpy(), compute_uv=False).round(6))

# (e) near-duplicate pairs among everything that could be an input (Spearman, no scipy needed)
CAND = SHARE_AREA + ["titled_share_luas", "mean_parcel_m2_log10", "parcel_density_log10", "coverage_log10",
                     "coverage_clip1", "luas_desa_log10", "age_median"]
c = x[CAND].corr(method="spearman").abs()
pairs = [(a, b, round(c.loc[a, b], 3)) for i, a in enumerate(CAND) for b in CAND[i + 1:] if c.loc[a, b] >= 0.9]
pd.DataFrame(pairs, columns=["column_a", "column_b", "abs_spearman"]).sort_values("abs_spearman", ascending=False)

In [ ]:
# --- 4-3. Rare rights: group them so the composition has fewer, steadier parts ---------------------------
rights = pd.DataFrame({
    "area_share_%": [100 * desa.loc[ok, f"{r}_luas"].sum() / desa.loc[ok, "luas_persil"].sum() for r in RIGHTS],
    "villages_present_%": [100 * (desa.loc[ok, f"{r}_luas"] > 0).mean() for r in RIGHTS],
    "parcel_share_%": [100 * desa.loc[ok, f"{r}_jumlah"].sum() / desa.loc[ok, "jumlah_persil"].sum() for r in RIGHTS],
}, index=RIGHTS).round(2)

GROUPED = [r for r in RIGHTS if rights.loc[r, "area_share_%"] < 100 * RARE_AREA_SHARE]
KEPT = [r for r in RIGHTS if r not in GROUPED]
rights["decision"] = ["grouped into other" if r in GROUPED else "kept" for r in RIGHTS]
display(rights)

add_cols({"other_share_luas": desa[[f"{r}_share_luas" for r in GROUPED]].sum(axis=1, min_count=1)}, group=None)
print("kept separately:", KEPT)
print("grouped into 'other':", GROUPED, "| HGU is kept despite being present in few villages: it holds a large share of the area")

In [ ]:
# --- 4-4. Transform: sqrt of the shares (Hellinger geometry) + log columns ----------------------------------
# sqrt makes Euclidean distance between share vectors proportional to the Hellinger distance, and it tolerates zeros.
parts = [f"{r}_share_luas" for r in KEPT] + (["other_share_luas"] if GROUPED else [])
T = pd.DataFrame(index=desa.index)
for p in parts:
    T["s_" + p.replace("_share_luas", "")] = np.sqrt(desa[p])
T["log_parcel_density"] = desa["parcel_density_log10"]
T["log_mean_parcel"] = desa["mean_parcel_m2_log10"]
T["log_luas_desa"] = desa["luas_desa_log10"]
T["log1p_age_median"] = np.log1p(desa["age_median"])      # age is right-skewed (tail to 60+ years)

COMP_COLS = [c for c in T.columns if c.startswith("s_")]
SD_COLS = ["log_parcel_density", "log_mean_parcel"]
EXTRA_B = ["log_luas_desa", "log1p_age_median"]
COLS = {"A": COMP_COLS + SD_COLS, "B": COMP_COLS + SD_COLS + EXTRA_B}

# fit sets: analysis rows where every column of the variant is defined
FIT = {v: ok & T[cols].notna().all(axis=1) for v, cols in COLS.items()}
print("variant A columns:", COLS["A"])
print("variant B adds   :", EXTRA_B)
print({v: int(m.sum()) for v, m in FIT.items()}, "villages in the fit sets")
T.loc[FIT["A"], COLS["B"]].describe().T[["count", "min", "50%", "mean", "max"]].round(3)

In [ ]:
# --- 4-5. Who is NOT in the fit set, and why ("place later") ---------------------------------------------------
excl_reason = desa.index.map(excluded["reason"]).to_series(index=desa.index).fillna("")
reason_A = np.where(~ok, "excluded in Step 3: " + excl_reason,
            np.where(desa["parcel_status"] == "no_parcels", "no parcels: shares undefined", ""))
reason_B = np.where(reason_A != "", reason_A,
            np.where(~FIT["B"], "no registration year (no titled parcel): age undefined", ""))
place_later = pd.DataFrame({"reason_A": reason_A, "reason_B": reason_B}, index=desa.index)
place_later = place_later[(place_later["reason_A"] != "") | (place_later["reason_B"] != "")]

summary = pd.DataFrame({
    "A": place_later["reason_A"].replace("", "(in fit set)").str.replace(r"excluded in Step 3: .*", "excluded in Step 3", regex=True).value_counts(),
    "B": place_later["reason_B"].replace("", "(in fit set)").str.replace(r"excluded in Step 3: .*", "excluded in Step 3", regex=True).value_counts(),
}).fillna(0).astype(int)
summary.loc["(in fit set)"] = [int(FIT["A"].sum()), int(FIT["B"].sum())]
print("every village is either in a fit set or in place_later:",
      bool((FIT["A"].sum() + (place_later["reason_A"] != "").sum()) == len(desa)))
summary

In [ ]:
# --- 4-6. Scaling: fit on the fit set, store everything needed to re-apply it ------------------------------------
# * log / age columns: robust z-score (median, IQR / 1.349); falls back to the std if the IQR is 0
# * composition block: centered, then multiplied by ONE common factor so the block carries one unit of variance
#   per degree of freedom (6 parts that sum to 1 = 5 independent directions). Rare-right columns are NOT
#   inflated to unit variance each, which z-scoring every column would do.
def fit_recipe(variant):
    Tf = T.loc[FIT[variant], COLS[variant]]
    comp = [c for c in COLS[variant] if c in COMP_COLS]
    other = [c for c in COLS[variant] if c not in COMP_COLS]
    center, scale = {}, {}
    for c in other:
        q1, q3 = Tf[c].quantile([0.25, 0.75])
        center[c] = float(Tf[c].median())
        scale[c] = float((q3 - q1) / 1.349) or float(Tf[c].std())
    return {"comp_cols": comp, "other_cols": other, "center": center, "scale": scale,
            "comp_center": {c: float(Tf[c].mean()) for c in comp},
            "comp_dof": len(comp) - 1,
            "comp_factor": float(np.sqrt((len(comp) - 1) / Tf[comp].var().sum()))}

def apply_recipe(Tdf, recipe):
    """Scale any rows of T (fit set or place-later rows) with a fitted recipe; undefined values stay NaN."""
    X = pd.DataFrame(index=Tdf.index)
    for c in recipe["comp_cols"]:
        X[c] = (Tdf[c] - recipe["comp_center"][c]) * recipe["comp_factor"]
    for c in recipe["other_cols"]:
        X[c] = (Tdf[c] - recipe["center"][c]) / recipe["scale"][c]
    return X[COLS[recipe["variant"]]]

RECIPE = {v: {**fit_recipe(v), "variant": v} for v in COLS}

X_A = T.loc[FIT["A"], COLS["A"]].set_axis(desa.loc[FIT["A"], "kode8"])
X_B = T.loc[FIT["B"], COLS["B"]].set_axis(desa.loc[FIT["B"], "kode8"])
X_A_scaled = apply_recipe(T.loc[FIT["A"]], RECIPE["A"]).set_axis(desa.loc[FIT["A"], "kode8"])
X_B_scaled = apply_recipe(T.loc[FIT["B"]], RECIPE["B"]).set_axis(desa.loc[FIT["B"], "kode8"])

var_share = pd.DataFrame({
    "variance_unscaled": X_A.var(),
    "variance_scaled": X_A_scaled.var(),
})
var_share["share_of_total_scaled_%"] = (100 * var_share["variance_scaled"] / var_share["variance_scaled"].sum()).round(1)
print({v: (round(r["comp_factor"], 3), r["comp_dof"]) for v, r in RECIPE.items()}, "<- composition factor, degrees of freedom")
print("composition block share of total variance (A, scaled): %.1f%%" % var_share.loc[COMP_COLS, "share_of_total_scaled_%"].sum())
var_share.round(3)

In [ ]:
# --- 4-7. Evidence: how much does each choice actually change the geometry? ------------------------------------
# Compare pairwise distances between 1,500 random villages under each alternative (rank correlation).
# 1.00 = identical neighbourhood structure, much lower = the choice matters.
def pair_dist(X):
    X = np.asarray(X, dtype=float)
    g = X @ X.T
    sq = np.diag(g)
    d2 = sq[:, None] + sq[None, :] - 2 * g
    return np.sqrt(np.maximum(d2[np.triu_indices(len(X), 1)], 0))

def rank_corr(a, b):          # Spearman without scipy: Pearson on ranks
    return float(np.corrcoef(pd.Series(a).rank().to_numpy(), pd.Series(b).rank().to_numpy())[0, 1])

def clr(S, repl):             # centred log-ratio; zeros replaced by `repl`, then re-closed to sum 1
    S = np.where(S == 0, repl, S)
    S = S / S.sum(axis=1, keepdims=True)
    L = np.log(S)
    return L - L.mean(axis=1, keepdims=True)

rng = np.random.default_rng(SEED)
rows_A = rng.choice(desa.index[FIT["A"].to_numpy()], SAMPLE_N, replace=False)
share6 = desa.loc[rows_A, parts].to_numpy()
count6 = np.column_stack([desa.loc[rows_A, f"{r}_share_jumlah"] for r in KEPT] +
                         ([desa.loc[rows_A, [f"{r}_share_jumlah" for r in GROUPED]].sum(axis=1)] if GROUPED else []))
share8 = desa.loc[rows_A, SHARE_AREA].to_numpy()

base = pair_dist(np.sqrt(share6))                       # default composition geometry
ev = []
def add_ev(decision, alternative, d, note=""):
    ev.append({"decision": decision, "alternative": alternative, "rho_vs_default": round(rank_corr(base, d), 3), "note": note})

add_ev("share transform", "raw shares (no transform)", pair_dist(share6))
for repl in (1e-4, 1e-3, 1e-2):
    add_ev("share transform", f"CLR, zeros replaced by {repl:g}", pair_dist(clr(share6, repl)))
clr_lo, clr_hi = pair_dist(clr(share6, 1e-4)), pair_dist(clr(share6, 1e-2))
ev.append({"decision": "share transform", "alternative": "CLR 1e-4 versus CLR 1e-2 (each other)",
           "rho_vs_default": round(rank_corr(clr_lo, clr_hi), 3), "note": "how arbitrary the zero replacement is"})
add_ev("rare-right grouping", "no grouping (8 parts)", pair_dist(np.sqrt(share8)))
add_ev("what the shares measure", "parcel counts instead of area", pair_dist(np.sqrt(count6)))

# scaling schemes on the full variant-A matrix
Tf = T.loc[FIT["A"], COLS["A"]]
def comp_share(Xs):
    return round(100 * Xs[COMP_COLS].var().sum() / Xs.var().sum(), 1)
def scheme_comp_weight(w):       # composition block with total variance w, robust-z for the log columns
    r = RECIPE["A"]; f = np.sqrt(w / Tf[COMP_COLS].var().sum())
    Xs = pd.DataFrame({c: (Tf[c] - r["comp_center"][c]) * f for c in COMP_COLS})
    for c in SD_COLS:
        Xs[c] = (Tf[c] - r["center"][c]) / r["scale"][c]
    return Xs
schemes = {
    "default: 1 unit of variance per degree of freedom": X_A_scaled.set_axis(Tf.index),
    "z-score every column": (Tf - Tf.mean()) / Tf.std(),
    "each concept counts once (composition = 1)": scheme_comp_weight(1.0),
    "composition and log block equal": scheme_comp_weight(scheme_comp_weight(1.0)[SD_COLS].var().sum()),
    "no scaling": Tf,
}
def sample_dist(Xs):
    return pair_dist(Xs.loc[rows_A].to_numpy())
base_scaled = sample_dist(schemes["default: 1 unit of variance per degree of freedom"])
for name, Xs in list(schemes.items())[1:]:
    ev.append({"decision": "scaling", "alternative": name, "rho_vs_default": round(rank_corr(base_scaled, sample_dist(Xs)), 3),
               "note": f"composition = {comp_share(Xs)}% of total variance (default {comp_share(schemes['default: 1 unit of variance per degree of freedom'])}%)"})

# scope: variant B and composition-only, compared with variant A on villages present in both fit sets
rows_B = rng.choice(desa.index[FIT["B"].to_numpy()], SAMPLE_N, replace=False)
XA_s = apply_recipe(T.loc[rows_B], RECIPE["A"])
XB_s = apply_recipe(T.loc[rows_B], RECIPE["B"])
dA = pair_dist(XA_s.to_numpy())
ev.append({"decision": "scope", "alternative": "variant B (adds village size + registration age)",
           "rho_vs_default": round(rank_corr(dA, pair_dist(XB_s.to_numpy())), 3), "note": "A versus B on the same villages"})
ev.append({"decision": "scope", "alternative": "composition only (drop parcel size / density)",
           "rho_vs_default": round(rank_corr(dA, pair_dist(XA_s[COMP_COLS].to_numpy())), 3), "note": "A versus its composition block alone"})

evidence = pd.DataFrame(ev)
EVID = {(r["decision"], r["alternative"]): r["rho_vs_default"] for r in ev}
evidence

In [ ]:
# --- 4-8. Readiness checks: this must pass before anything is exported ------------------------------------------------
matrices = {"X_A": X_A, "X_A_scaled": X_A_scaled, "X_B": X_B, "X_B_scaled": X_B_scaled}
checks = pd.DataFrame({
    name: {
        "rows": len(M), "columns": M.shape[1],
        "NaN values": int(M.isna().sum().sum()),
        "infinite values": int(np.isinf(M.to_numpy()).sum()),
        "non-numeric columns": int((~M.dtypes.map(lambda t: np.issubdtype(t, np.number))).sum()),
        "duplicate village ids": int(M.index.duplicated().sum()),
        "constant columns": int((M.std() == 0).sum()),
        "exact duplicate rows": int(M.duplicated().sum()),
    } for name, M in matrices.items()
}).T
display(checks)
bad = checks[["NaN values", "infinite values", "non-numeric columns", "duplicate village ids", "constant columns"]].sum().sum()
assert bad == 0, "readiness check failed: see the table above"
print("hard checks passed (exact duplicate rows are informational: villages with identical profiles)")

# near-duplicate columns among the final inputs
cc = X_B_scaled.corr(method="spearman").abs()
cols_b = list(cc.columns)
dups = [(a, b, round(cc.loc[a, b], 3)) for i, a in enumerate(cols_b) for b in cols_b[i + 1:] if cc.loc[a, b] >= 0.9]
print("input pairs with |Spearman| >= 0.9:", dups if dups else "none")

# shape of each input after the transform
shape = pd.DataFrame({
    "skew": X_B.skew(),
    "pct_exact_zero": (100 * (X_B == 0).mean()).round(1),
    "pct_at_minimum": (100 * (X_B == X_B.min()).mean()).round(1),
    "p01_scaled": X_B_scaled.quantile(0.01),
    "p99_scaled": X_B_scaled.quantile(0.99),
    "n_beyond_5_sd": (X_B_scaled.abs() > 5).sum(),
}).round(2)
shape

In [ ]:
# --- 4-8b. See it: the inputs after the transform, and a first look at structure -----------------------------------------
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, c in zip(axes.ravel(), COLS["B"]):
    ax.hist(X_B[c], bins=60, color="tab:blue")
    ax.set_title(c, fontsize=10)
    ax.set_yscale("log")
plt.suptitle("Inputs after the transform (log y-axis): spikes at 0 are villages that do not hold that right", y=1.02)
plt.tight_layout()
plt.show()

# PCA by SVD (numpy only): how many directions carry the variance, and is there visible structure?
Z = X_A_scaled.to_numpy() - X_A_scaled.to_numpy().mean(axis=0)
U, S, Vt = np.linalg.svd(Z, full_matrices=False)
explained = S ** 2 / (S ** 2).sum()
pcs = Z @ Vt[:2].T
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].bar(range(1, len(explained) + 1), 100 * explained, color="tab:gray")
ax[0].plot(range(1, len(explained) + 1), 100 * np.cumsum(explained), "o-", color="tab:red")
ax[0].set_xlabel("principal component"); ax[0].set_ylabel("% of variance (bars) / cumulative (line)")
ax[0].set_title("variant A, scaled: variance per component")
lo, hi = np.percentile(pcs, [0.5, 99.5], axis=0)
hb = ax[1].hexbin(pcs[:, 0], pcs[:, 1], gridsize=70, bins="log", mincnt=1, extent=(lo[0], hi[0], lo[1], hi[1]), cmap="viridis")
fig.colorbar(hb, ax=ax[1], label="villages (log)")
ax[1].set_xlabel("PC1"); ax[1].set_ylabel("PC2"); ax[1].set_title("variant A, scaled: first two components")
plt.tight_layout()
plt.show()
print("variance explained by PC1..PC4 (%):", (100 * explained[:4]).round(1))

In [ ]:
# --- 4-9. Decision log ------------------------------------------------------------------------------------------------------
def e(decision, alternative):
    return EVID[(decision, alternative)]

fitA = FIT["A"]
skew_raw = desa.loc[fitA, ["mean_parcel_m2", "parcel_density"]].skew().round(1)
skew_log = desa.loc[fitA, ["mean_parcel_m2_log10", "parcel_density_log10"]].skew().round(2)
try:
    overshoot_effect = f"dropping the overshoot rows changes median parcel density by {delta.loc['median parcel_density', 'base + drop overshoot rows']}% (Step 3b-9)"
except Exception:
    overshoot_effect = "see Step 3b-9"
dof = RECIPE["A"]["comp_dof"]
no_par = ok & (desa["parcel_status"] == "no_parcels")
def robust_z(s):
    q1, q3 = s.quantile([0.25, 0.75])
    return (s - s.median()) / ((q3 - q1) / 1.349)
age_raw = desa.loc[FIT["B"], "age_median"]
age_log = np.log1p(age_raw)
scal = evidence[evidence["decision"] == "scaling"]
scaling_text = "; ".join(f"{a}: rho {r} ({n})" for a, r, n in zip(scal["alternative"], scal["rho_vs_default"], scal["note"]))

decision_log = pd.DataFrame([
    ("D1", "Scope of the inputs", "Variant A: land-tenure profile only. Variant B: A + village size + registration age.",
     "A only; B only; everything",
     "A keeps clusters interpretable as tenure types and keeps flood / RDTR / years / geography free to validate them. B is offered so the effect of extras can be measured.",
     f"distance rank correlation A versus B = {e('scope', 'variant B (adds village size + registration age)')}; composition block alone versus A = {e('scope', 'composition only (drop parcel size / density)')}"),
    ("D2", "Village area as an input", "Not in A; log10 area in B.",
     "include in both; exclude from both", "Area is extremely skewed and would dominate distances; tenure type should not just be village size.",
     f"B differs from A by rho = {e('scope', 'variant B (adds village size + registration age)')}"),
    ("D3", "What the shares measure", "Share of parcel AREA, not of parcel count.",
     "parcel counts; both", "Area shows what the land is used for (HGU is 0.02% of parcels but about 17% of the area); counts mostly say how subdivided a village is, and parcel size / density already cover that.",
     f"distance rank correlation, count shares versus area shares = {e('what the shares measure', 'parcel counts instead of area')}"),
    ("D4", "Rare rights", f"Rights below {RARE_AREA_SHARE:.0%} of total area are grouped into 'other': {', '.join(GROUPED)}.",
     "keep all 8 parts", "Fewer, steadier parts; the rare ones are zero in almost every village. HGU is kept separate because it holds a large share of the area.",
     f"distance rank correlation, ungrouped versus grouped = {e('rare-right grouping', 'no grouping (8 parts)')}: the geometry is unchanged"),
    ("D5", "Transform for the shares", "Square root (Hellinger geometry).",
     "raw shares; centred log-ratio (CLR) with zero replacement",
     "Shares sum to 1, so raw distances are distorted; CLR cannot handle the many exact zeros without an arbitrary replacement value; sqrt tolerates zeros and needs no parameter.",
     f"raw shares rho = {e('share transform', 'raw shares (no transform)')}; CLR rho = {e('share transform', 'CLR, zeros replaced by 0.0001')} / {e('share transform', 'CLR, zeros replaced by 0.001')} / {e('share transform', 'CLR, zeros replaced by 0.01')} for replacement 1e-4 / 1e-3 / 1e-2; two CLR variants versus each other = {e('share transform', 'CLR 1e-4 versus CLR 1e-2 (each other)')}"),
    ("D6", "Redundant columns", "Dropped: titled share, coverage (log, clipped), count-based shares, per-right mean sizes.",
     "keep them", f"titled share = 1 - kosong share (max deviation {dev_titled:.0e}); log10 density + log10 mean parcel size = log10 coverage + 4 (max deviation {dev_log:.0e}), so coverage adds no information and a third column would make the matrix rank-deficient.",
     "exact identities, see 4-2"),
    ("D7", "Parcel size and density", "log10 of both.",
     "raw values", "Both are extremely right-skewed; the log removes the tail without needing a cap.",
     f"skew of mean parcel size {skew_raw['mean_parcel_m2']} -> {skew_log['mean_parcel_m2_log10']}; parcel density {skew_raw['parcel_density']} -> {skew_log['parcel_density_log10']}"),
    ("D8", "Villages with no parcels", f"Not fitted ({int(no_par.sum()):,} analysis rows); listed in place_later.",
     "impute shares; drop silently", "Shares are undefined (0/0); imputing would invent a tenure profile. They are a distinct, informative group.",
     f"see 4-5; {100 * desa.loc[no_par, 'flood_all_zero'].mean():.0f}% of them have zero flood area (all analysis rows: {100 * desa.loc[ok, 'flood_all_zero'].mean():.0f}%) and {100 * (desa.loc[no_par, 'rdtr_share'] > 0).mean():.0f}% have an RDTR (all analysis rows: {100 * (desa.loc[ok, 'rdtr_share'] > 0).mean():.0f}%)"),
    ("D9", "Rows excluded in Step 3", "Not fitted; listed in place_later with the reason.",
     "fit them; drop silently", "Placeholders and zero-area villages have no meaningful profile; for ratio outliers the area-based features are unreliable.",
     "see 4-5"),
    ("D10", "Overshoot rows (parcels 1-10x village area)", "Kept in the fit set.",
     "drop them", "They look like ordinary, denser villages with overlapping parcels; dropping them biases the sample toward sparse villages.",
     overshoot_effect),
    ("D11", "Scaling", f"Robust z-score for the log / age columns; composition block scaled jointly to {dof} units of variance ({dof} = parts - 1).",
     "z-score every column; each concept counts once; no scaling",
     "Z-scoring every column would inflate the rare-right columns to the same weight as common ones; one unit per degree of freedom keeps the geometry honest.",
     scaling_text),
    ("D12", "Flood, RDTR, years, geography", "Interpretation only; never used to build clusters.",
     "use as inputs", "Zero flood area is ambiguous and strongly regional; RDTR exists for ~17% of villages; years are undefined for untitled villages; geography should be an outcome, not an input.",
     "Step 3b-4 (flood), Step 3b-3 (concentration)"),
    ("D13", "Binary has_* flags", "Not used.",
     "add as inputs", "Each flag is already visible in the matching share (a zero share = flag false).",
     "by construction"),
    ("D14", "Registration age (variant B)", "log(1 + years since the median registration year).",
     "raw years; square root", "Raw age has a long right tail (up to 60+ years) that would dominate distances for a few percent of villages.",
     f"skew {age_raw.skew():.2f} -> {age_log.skew():.2f}; villages beyond 5 robust SD: {int((robust_z(age_raw).abs() > 5).sum()):,} -> {int((robust_z(age_log).abs() > 5).sum()):,}"),
], columns=["id", "decision", "default", "alternatives_considered", "reason", "evidence"])

for _, r in decision_log.iterrows():          # printed as text: long reasons would be truncated in a table
    print(f"{r['id']}  {r['decision']}")
    print("    default     :", r["default"])
    print("    alternatives:", r["alternatives_considered"])
    print("    reason      :", r["reason"])
    print("    evidence    :", r["evidence"])

In [ ]:
# --- 4-10. Package and export ------------------------------------------------------------------------------------------------
import json, sys

village_ids = desa[["kode8", "kode", "nama", "kecamatan", "kabupaten", "provinsi", "tipe", "kab_key"]].set_index("kode8")

INTERP = (["luas_desa", "jumlah_persil", "luas_persil", "coverage", "titled_share_luas", "parcel_status"]
          + SHARE_AREA + ["other_share_luas"]
          + ["flood_low_share", "flood_mid_share", "flood_high_share", "flood_any_share", "flood_all_zero", "rdtr_share"]
          + ["tahun_min", "tahun_median", "tahun_max", "year_span", "age_first", "age_median", "age_last"]
          + ["is_overshoot", "is_zero_parcels", "prov_code_mismatch", "is_minor_rule_break", "analysis_ok"])
interp = desa[["kode8"] + INTERP].set_index("kode8")

place_later_out = place_later.copy()
place_later_out.index = desa.loc[place_later.index, "kode8"]
place_later_T = T.loc[place_later.index, COLS["B"]].set_axis(place_later_out.index)
scaled_A_later = apply_recipe(T.loc[place_later.index], RECIPE["A"]).set_axis(place_later_out.index).add_suffix("__scaled_A")
place_later_out = pd.concat([place_later_out, place_later_T, scaled_A_later], axis=1)

recipe_json = {
    "created": str(pd.Timestamp.now().round("s")),
    "environment": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__},
    "parameters": {"RARE_AREA_SHARE": RARE_AREA_SHARE, "RATIO_CAP": RATIO_CAP, "ZERO_THRESH": ZERO_THRESH,
                   "REF_YEAR": REF_YEAR, "STRAY_MAX": STRAY_MAX, "SEED": SEED},
    "rights": {"kept_separately": KEPT, "grouped_into_other": GROUPED},
    "transform": "s_<right> = sqrt(area share of the right); log_parcel_density = log10(parcels per hectare); "
                 "log_mean_parcel = log10(parcel area / parcel count in m2); log_luas_desa = log10(village area in m2); "
                 "log1p_age_median = log(1 + REF_YEAR - tahun_median)",
    "columns": COLS,
    "scaling": {v: {k: val for k, val in r.items() if k != "variant"} for v, r in RECIPE.items()},
    "fit_set_sizes": {v: int(m.sum()) for v, m in FIT.items()},
}

files = {
    "X_A.csv": X_A, "X_A_scaled.csv": X_A_scaled, "X_B.csv": X_B, "X_B_scaled.csv": X_B_scaled,
    "village_ids.csv": village_ids, "interpretation_columns.csv": interp, "place_later.csv": place_later_out,
    "excluded_rows.csv": excluded.set_index("kode8"), "decision_log.csv": decision_log.set_index("id"),
}
deliverables = pd.DataFrame({"rows": {k: len(v) for k, v in files.items()}, "columns": {k: v.shape[1] for k, v in files.items()}})
if EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    for name, frame in files.items():
        frame.to_csv(EXPORT_DIR / name, float_format="%.6g")
    (EXPORT_DIR / "transform_recipe.json").write_text(json.dumps(recipe_json, indent=2))
    print("data files written to", EXPORT_DIR, "(README.md and BACKGROUND.md are written by the next cell)")
else:
    print("EXPORT = False: nothing written")
deliverables

In [ ]:
# --- 4-11. Hand-off documents: README.md (entry point) and BACKGROUND.md (history and findings) ------------------
# Written from this notebook's own variables, so the numbers in the documents cannot drift from the data.
NL = chr(10)

def cell(v):
    """Make a value safe inside a markdown table cell."""
    return str(v).replace("|", "/").replace(NL, " ")

def rho(decision, alternative):
    return EVID[(decision, alternative)]

snapshot = desa["dihitung_pada"].dt.date.max()
n_total, nA, nB, n_later = len(desa), int(FIT["A"].sum()), int(FIT["B"].sum()), len(place_later)
env = recipe_json["environment"]
no_par = ok & (desa["parcel_status"] == "no_parcels")
no_par_top = ", ".join(f"{p} ({n})" for p, n in desa.loc[no_par, "provinsi"].value_counts().head(4).items())
var_prov, var_kab = share_explained("provinsi"), share_explained("kab_code")
pc12 = 100 * explained[:2].sum()
scaling_rows = evidence[evidence["decision"] == "scaling"]

file_notes = {
    "README.md": "this file: start here",
    "BACKGROUND.md": "how the data was prepared, what was found, what is still unresolved",
    "X_A.csv": "variant A inputs (land-tenure profile only), unscaled",
    "X_A_scaled.csv": "variant A inputs, scaled: ready for distance-based methods",
    "X_B.csv": "variant B inputs (A + village size + registration age), unscaled",
    "X_B_scaled.csv": "variant B inputs, scaled",
    "village_ids.csv": "names and keys for every village (raw names, inconsistent casing)",
    "interpretation_columns.csv": "raw shares, flood, RDTR, years, flags: for profiling clusters afterwards (village size and age enter variant B only in transformed form)",
    "place_later.csv": "villages outside a fit set, with the reason (scaled values are shown where defined, but are unreliable for excluded rows)",
    "excluded_rows.csv": "rows removed in Step 3 (placeholders, zero area, area outliers)",
    "decision_log.csv": "every preparation decision: default, alternatives, reason, measured evidence",
    "transform_recipe.json": "every parameter needed to reproduce or re-apply the transform",
}

# =========================================================================== README.md
R = []
R += [
    "# Clustering-ready data: desa (Indonesian villages, land tenure)",
    "",
    "## Status",
    "",
    "- **Data preparation is finished and documented. Modelling has not started.** Choosing the algorithm, the number of clusters,",
    "  validating and naming clusters is the next stage and belongs to the modelling group.",
    f"- Source: `desa.xlsx` (not included in this folder), snapshot computed {snapshot}. {n_total:,} villages (desa and kelurahan), one row per village.",
    f"- Variant A (default): {nA:,} villages x {len(COLS['A'])} columns. Variant B: {nB:,} villages x {len(COLS['B'])} columns.",
    f"- {n_later:,} villages are not in a fit set and are listed in `place_later.csv` with the reason.",
    "- Read `BACKGROUND.md` for the story behind the data, and `decision_log.csv` for the reason and evidence behind every choice.",
    "",
    "## What is in this folder",
    "",
    "| file | rows | columns | what it is |",
    "|---|---|---|---|",
]
for f, note in file_notes.items():
    rows_cols = (f"{int(deliverables.loc[f, 'rows']):,}", str(int(deliverables.loc[f, 'columns']))) if f in deliverables.index else ("-", "-")
    R.append(f"| `{f}` | {rows_cols[0]} | {rows_cols[1]} | {note} |")

R += [
    "",
    "## Quick start (Python)",
    "",
    "```python",
    "import pandas as pd",
    "",
    "d = 'clustering_ready/'                                   # path to this folder",
    "T = dict(index_col='kode8', dtype={'kode8': str})         # kode8 is TEXT: it has a leading zero",
    "X     = pd.read_csv(d + 'X_A_scaled.csv', **T)            # model matrix, variant A, scaled",
    "ids   = pd.read_csv(d + 'village_ids.csv', **T)           # names and kabupaten key",
    "meta  = pd.read_csv(d + 'interpretation_columns.csv', **T)  # for profiling clusters only",
    "later = pd.read_csv(d + 'place_later.csv', **T)           # villages outside a fit set; reason_A / reason_B say why",
    "",
    "# ... fit a model on X, then attach the labels and profile the clusters:",
    "# labels  = pd.Series(model_labels, index=X.index, name='cluster')",
    "# profile = meta.join(labels, how='inner').groupby('cluster').median(numeric_only=True)",
    "",
    "# who is missing from variant A, and why (an empty reason_A means the village IS in X_A):",
    "print(later['reason_A'].dropna().str.slice(0, 40).value_counts())",
    "```",
    "",
    "To rebuild or re-apply the scaling, read `transform_recipe.json` (`scaling` -> `A` or `B`):",
    "composition columns = `(value - comp_center) * comp_factor`; the other columns = `(value - center) / scale`.",
    "",
    "## Who is in `place_later.csv`, and what to do with them",
    "",
    "| group | variant A | variant B | suggested handling |",
    "|---|---|---|---|",
    f"| no parcels: shares undefined | {summary.loc['no parcels: shares undefined', 'A']:,} | {summary.loc['no parcels: shares undefined', 'B']:,} | no tenure profile exists: report them as a group of their own, do not assign them to a cluster |",
    f"| excluded in Step 3 | {summary.loc['excluded in Step 3', 'A']:,} | {summary.loc['excluded in Step 3', 'B']:,} | placeholders, zero area and area outliers: areas are unreliable, so do not assign them |",
    f"| no registration year (age undefined) | {summary.loc['no registration year (no titled parcel): age undefined', 'A']:,} | {summary.loc['no registration year (no titled parcel): age undefined', 'B']:,} | already in `X_A`; for variant B they can be scored on the A columns if needed |",
    "",
    "## The input columns",
    "",
    f"- `{'`, `'.join(COMP_COLS)}`: square root of the share of the village's parcel **area** held under each right",
    "  (the square root makes Euclidean distance proportional to the Hellinger distance and tolerates exact zeros).",
    f"  `s_other` groups the rights below {RARE_AREA_SHARE:.0%} of total area: {', '.join(GROUPED)}.",
    "- `log_parcel_density`: log10 of parcels per hectare. `log_mean_parcel`: log10 of the mean parcel size in m2.",
    f"- Variant B adds `log_luas_desa` (log10 village area, m2) and `log1p_age_median` (log of 1 + years between {REF_YEAR} and the median registration year).",
    f"- Scaled files: log / age columns are robust z-scores; the composition block is centred and multiplied by one common factor so it carries {RECIPE['A']['comp_dof']} units of variance (6 parts that sum to 1 = {RECIPE['A']['comp_dof']} independent directions).",
    "",
    "## Rules for working with this data",
    "",
    "1. Read `kode8` as **text**. Spreadsheets and default CSV readers drop its leading zero.",
    "2. Use the `*_scaled.csv` files as delivered for distance-based methods. Do not rescale the blocks again by habit; if you change the weighting,",
    "   do it on purpose (decision D11) and record it.",
    "3. Do **not** use columns from `interpretation_columns.csv` as clustering inputs without an explicit new decision. Flood in particular (see Caveats).",
    "   The only vetted extras, village size and registration age, are already inside `X_B` in transformed form.",
    "4. Do not drop or impute `place_later` villages silently. Villages with no parcels have no tenure profile (the shares are 0/0): report them as their own group.",
    "5. Any change to the preparation is a new decision: add an entry (D15, D16, ...) with the reason and the measured effect.",
    "6. The evidence numbers describe how much a preparation choice changes distances. They are **not** a measure of clustering quality.",
    "",
    "## Decisions already made (reasons and evidence: `decision_log.csv`)",
    "",
    "| id | decision | default |",
    "|---|---|---|",
]
for _, r in decision_log.iterrows():
    R.append(f"| {r['id']} | {cell(r['decision'])} | {cell(r['default'])} |")

R += [
    "",
    "## Decisions the group still has to make",
    "",
    f"1. **Variant A or B.** B differs from A by a distance rank correlation of {rho('scope', 'variant B (adds village size + registration age)')} (1.00 = identical neighbourhoods).",
    "   A is the default because it keeps clusters readable as tenure types; B is there so the effect of extra columns can be tested.",
    f"2. **Shares of area or of parcel count.** The default measures area. Parcel-count shares give a clearly different geometry (rank correlation {rho('what the shares measure', 'parcel counts instead of area')}).",
    "   A variant built on counts is not delivered; cells 4-4 and 4-7 of the notebook show how it would be made.",
    "3. **Weight of the composition block.** The default gives it " + f"{comp_share(schemes['default: 1 unit of variance per degree of freedom'])}% of the total variance. Alternatives, as rank correlation with the default:",
]
for _, r in scaling_rows.iterrows():
    R.append(f"   - {r['alternative']}: {r['rho_vs_default']}; {r['note'].replace(' (default ', ', default ').rstrip(')')})")
R += [
    "4. **Whether to add flood, RDTR or registration years later**, once the flood question is settled.",
    "",
    "## Caveats you should know before interpreting anything",
    "",
    f"- **Flood zeros are ambiguous.** {100 * desa['flood_all_zero'].mean():.1f}% of villages have no flood area in any class: 'no hazard' or 'not computed' cannot be told apart.",
    f"  The zeros are strongly regional (kabupaten explains {100 * var_kab:.0f}% of the variance of the zero flag, province {100 * var_prov:.0f}%), and {len(flood_suspect_kab)} kabupaten have 90% or more of their villages at zero.",
    "  Do not read flood differences between clusters as proven hazard differences.",
    f"- **A first look does not show clearly separated groups.** In a PCA of variant A (scaled), the first two components explain {pc12:.0f}% of the variance and the villages form a continuous cloud with dense regions.",
    "  Discrete clusters are not guaranteed; a gradient that an algorithm cuts into segments is a real possibility. Test this before presenting clusters as 'types'.",
    f"- **Villages with no parcels** ({int(no_par.sum()):,} in the analysis rows) are concentrated in {no_par_top}. They also have more zero-flood and less RDTR coverage than other villages,",
    "  which looks like a regional coverage gap (hypothesis, not confirmed).",
    f"- **Excluded rows** ({len(excluded)}): placeholders, villages with zero area, and {int(desa['is_ratio_outlier'].sum())} villages whose parcel area is more than {RATIO_CAP}x the village area. Their area-based values are unreliable even where `place_later.csv` shows them.",
    "- **Names are raw.** `village_ids.csv` keeps the source spelling (UPPER and Title case mixed). Group at kabupaten level with `kab_key`, not with the name:",
    f"  a name can hide a Kabupaten and a Kota. {int((desa.groupby(['provinsi_n', 'kabupaten_n'])['kab_key'].nunique() > 1).sum())} names still map to more than one key (mostly real Kabupaten/Kota twins; Kab. Bogor is split over two codes).",
    "- **Assumptions not yet confirmed with the data owner:** area units are m2; `kosong` = parcels with no registered right; `banjir_*` = areas in low / medium / high flood-hazard zones;",
    "  `rdtr_luas` = area covered by an RDTR plan; `tahun_*` = registration years of titled parcels.",
    f"- The evidence correlations were measured on a random sample of {SAMPLE_N:,} villages (seed {SEED}).",
    "",
    "## Suggested next steps for modelling (not prescriptive)",
    "",
    "1. Check whether the data has cluster structure at all (tendency tests, projections), given the continuous cloud above.",
    "2. Compare several algorithms and several criteria for the number of clusters. With about 78,000 rows, compute expensive metrics (such as the silhouette) on samples.",
    "3. Test stability: different seeds, bootstrap resamples, variant A versus B, and a changed composition weight.",
    "4. Test whether clusters merely reproduce geography (for example Cramer's V between cluster and province).",
    "5. Profile and name clusters with `interpretation_columns.csv` (raw rights shares, flood, RDTR, registration years, village type, geography).",
    "6. Report `place_later` explicitly: villages with no parcels as a group of their own; excluded rows are not assigned (unreliable areas); for variant B, the villages that only lack a registration year can be scored on the A columns.",
    "",
    "## Glossary",
    "",
    "- **desa / kelurahan**: rural village / urban ward. **kabupaten / kota**: regency / city. **provinsi**: province.",
    "- **persil**: land parcel. **hak milik**: freehold ownership title. **HGB** (hak guna bangunan): right to build. **HGU** (hak guna usaha): cultivation / plantation concession.",
    "- **hak pakai**: right of use. **hak pengelolaan**: management right over state land. **hak wakaf**: endowment (religious / charitable) land. **lainnya**: other.",
    "- **kosong**: 'empty'; here understood as parcels with no registered right (assumption, supported by the registration-year pattern).",
    "- **RDTR**: detailed spatial plan. **banjir**: flood. **kode8**: 8-digit village code PP KK CC DD (province, kabupaten/kota, kecamatan, village).",
    "",
    "## If you are an AI assistant helping a teammate",
    "",
    "- Read this README, `decision_log.csv` and `transform_recipe.json` before writing any code. Do not redo the preprocessing; use the delivered matrices.",
    "- Keep `kode8` as text. Do not use `interpretation_columns.csv` as model input unless the user explicitly decides to.",
    "- Do not present clusters as real or as 'types' before stability and structure checks; the first look suggests a continuum. State uncertainty plainly.",
    "- Do not describe flood differences as hazard evidence. Do not silently drop or impute `place_later` villages.",
    "- If a change to the preparation is needed, propose it as a new decision with its reason and a measured effect, and tell the user.",
    "- If something in this folder contradicts `BACKGROUND.md`, this README and `decision_log.csv` win.",
    "",
    "## Provenance and reproducibility",
    "",
    f"- Built by `master.ipynb`: Step 1 (structural relations), Step 2 (features), Step 3 (robust preparation), Step 3b (anomaly audit), Step 4 (cells 4-1 to 4-11). Run the notebook top to bottom with `desa.xlsx` in `data/`. Packages: numpy, pandas, matplotlib, and scipy (pandas uses it for Spearman correlations in earlier steps).",
    f"- Environment when built: Python {env['python']}, pandas {env['pandas']}, numpy {env['numpy']}. Parameters: `RARE_AREA_SHARE` {RARE_AREA_SHARE}, `RATIO_CAP` {RATIO_CAP}, `ZERO_THRESH` {ZERO_THRESH}, `STRAY_MAX` {STRAY_MAX}, `SEED` {SEED}.",
    f"- Created {recipe_json['created']}.",
]

# =========================================================================== BACKGROUND.md
B = []
B += [
    "# Background: how this data was prepared",
    "",
    "This is the story behind `clustering_ready`. For the rules of use and the open decisions, read `README.md` first.",
    "",
    "## 1. The dataset",
    "",
    f"- `desa.xlsx`: {n_total:,} villages (desa and kelurahan) x 33 columns, one row per village, computed {snapshot} in one batch (a snapshot, not a time series).",
    "- Per village: village area, the number and area of land parcels under each of 8 kinds of land right, flood-hazard areas (low / medium / high), the area covered by a detailed spatial plan (RDTR), and the minimum / median / maximum registration year.",
    "- The 8 rights are a complete partition: they add up exactly to the village's parcel count and parcel area in every row.",
    "",
    "## 2. Data quality: what was found and what was done",
    "",
    "| issue (flag name in the notebook) | rows | share of rows | treatment |",
    "|---|---|---|---|",
]
bg_label = {
    "is_pseudo": "placeholder records (not villages)",
    "is_zero_area": "village area is zero",
    "is_ratio_outlier": f"parcel area more than {RATIO_CAP}x the village area",
    "is_overshoot": f"parcel area between 1x and {RATIO_CAP}x the village area",
    "is_zero_parcels": "village has no parcels",
    "prov_code_mismatch": "province code disagrees with the province name",
    "flood_all_zero": "no flood area in any class",
    "is_minor_rule_break": "minor rule breaks (kosong count without area; titled parcels without a year)",
}
bg_treatment = {
    "is_pseudo": "excluded (not a village)",
    "is_zero_area": "excluded (every ratio is undefined)",
    "is_ratio_outlier": "excluded (almost certainly an error)",
    "is_overshoot": "kept in the fit set",
    "is_zero_parcels": "not fitted (shares undefined); listed in place_later as a group of its own",
    "prov_code_mismatch": "kept (label problem only)",
    "flood_all_zero": "kept, but flood is interpretation-only until the zeros are explained",
    "is_minor_rule_break": "kept (immaterial)",
}
for flag, r in anomalies.iterrows():
    B.append(f"| {bg_label[flag]} (`{flag}`) | {int(r['n_rows']):,} | {r['pct_rows']}% | {bg_treatment[flag]} |")

zero_area_top = desa.loc[desa["is_zero_area"], "provinsi"].value_counts()
out_mask = desa["is_ratio_outlier"]
B += [
    "",
    "- **Placeholder records.** Two rows are not villages: one named HGU (kode 33000000) and one whose name and kecamatan are '-' (kode 16030000). They are excluded.",
    f"- **Area outliers.** {int(out_mask.sum())} villages have a parcel area more than {RATIO_CAP}x the village area. They are scattered across provinces (the top province holds {100 * desa.loc[out_mask, 'provinsi'].value_counts(normalize=True).iloc[0]:.0f}%),",
    f"  {int((desa.loc[out_mask, 'kosong_share_luas'] >= 0.9).sum())} of them are at least 90% `kosong` land, and in {int((desa['outlier_culprit'] == 'parcel area (village looks normal)').sum())} the village area looks normal for its kabupaten,",
    "  so the parcel area is the likely inflated side (a heuristic, not proof).",
    f"- **Zero village area** ({int(desa['is_zero_area'].sum())} rows) is concentrated in {zero_area_top.index[0]} ({int(zero_area_top.iloc[0])} rows), which points to a local source problem.",
    f"- **Parcels larger than the village** (1x to {RATIO_CAP}x, {int(desa['is_overshoot'].sum()):,} rows) are kept: they look like ordinary, denser villages with overlapping parcels, and dropping them would bias the sample toward sparse villages.",
    f"- **Flood zeros.** {100 * desa['flood_all_zero'].mean():.1f}% of villages have no flood area. The pattern is regional (kabupaten explains {100 * var_kab:.0f}% of the variance, province {100 * var_prov:.0f}%) and is not just tiny villages missing a map.",
    f"  {len(flood_suspect_kab)} kabupaten have at least 90% of villages at zero, mostly highland, volcanic or small-island areas, so real zeros are plausible. It cannot be settled from this file.",
    f"- **Province code mismatches** ({int(desa['prov_code_mismatch'].sum())} rows) are a label problem only; the numbers are fine.",
    "- **Names and codes.** Casing is inconsistent; a kabupaten name can hide a Kabupaten and a Kota; a few kabupaten codes are shared by two names (many look like a regency and the one it was split from). `kab_key` combines province, normalized name and code, and merges stray legacy codes.",
    "",
    "## 3. Relationships that exist by construction (so they are not 'discoveries')",
    "",
    "- Parcel count and parcel area are the sums over the 8 rights; for every right the count is zero exactly when the area is zero.",
    "- Flood areas and the RDTR area never exceed the village area. A village has registration years exactly when it has at least one titled parcel, apart from 6 villages with titled parcels and no year.",
    "- Titled share = 1 - kosong share. And log10(parcels per hectare) + log10(mean parcel size) = log10(parcel area / village area) + 4 exactly, so coverage is not an independent third column.",
    "",
    "## 4. What the early analysis showed",
    "",
    f"- Village area and parcel count are almost unrelated (Spearman between {sens.loc['spearman luas_desa ~ jumlah_persil'].min():.3f} and {sens.loc['spearman luas_desa ~ jumlah_persil'].max():.3f} across every exclusion scenario): larger villages do not have proportionally more parcels.",
    f"- Medians and rank correlations barely move with the exclusion choices; means, sums and Pearson do (for example the Pearson correlation of village area and parcel area is {sens.loc['pearson luas_desa ~ luas_persil', 'base (analysis_ok)']:.2f} on the cleaned rows and {sens.loc['pearson luas_desa ~ luas_persil', 'no exclusions at all']:.2f} without exclusions).",
    "- All area columns are extremely right-skewed; the work uses medians, ranks and log scales.",
    "",
    "## 5. The pipeline in one page (`master.ipynb`)",
    "",
    "- **Step 1, structural relations:** a rule book of identities, inequalities and null patterns; the 8-digit `kode8`; normalized names; hierarchy checks.",
    f"- **Step 2, size-neutral features:** shares, mean sizes, density, coverage, flood and RDTR shares, years; undefined ratios are left as NaN, never imputed.",
    f"- **Step 3, robust preparation:** flags, `analysis_ok`, a cut-off sensitivity table, presence flags for zero-heavy columns, log columns. {int((~ok).sum())} rows are excluded from analysis.",
    "- **Step 3b, anomaly audit:** overlap and regional concentration of every anomaly, the flood-zero investigation, the area-outlier triage, `parcel_status`, `kab_key`, a sensitivity table.",
    "- **Step 4, clustering-ready data:** column roles, redundancy removal, rare-rights grouping, sqrt transform, fit / place-later split, scaling, evidence, readiness checks, decision log, export.",
    "",
    "## 6. Still unresolved",
    "",
    "- Whether zero flood area means 'no hazard' or 'not computed' (ask the data owner, or compare with an elevation layer).",
    "- The villages with no parcels, concentrated in a few provinces, and the zero-area villages concentrated in one province: coverage gaps or real?",
    "- The meaning of `kosong`, the units, and the exact definition of the flood and RDTR columns (see the assumptions in `README.md`).",
    "- Whether the data contains discrete clusters or a gradient (the first look suggests a continuum).",
    "",
    "## 7. How to reproduce",
    "",
    f"Open `master.ipynb`, put `desa.xlsx` in `data/`, run all cells from the top (about a minute; needs numpy, pandas, matplotlib and scipy). Step 4 writes this folder. Environment when built: Python {env['python']}, pandas {env['pandas']}.",
]

if EXPORT:
    (EXPORT_DIR / "README.md").write_text(NL.join(R) + NL, encoding="utf-8")
    (EXPORT_DIR / "BACKGROUND.md").write_text(NL.join(B) + NL, encoding="utf-8")
    missing = [f for f in file_notes if not (EXPORT_DIR / f).exists()]
    print("documents written to", EXPORT_DIR, "| files listed in the README that are missing:", missing if missing else "none")
    print(pd.Series({p.name: f"{p.stat().st_size / 1e6:.2f} MB" for p in sorted(EXPORT_DIR.iterdir())}).to_string())
else:
    print("EXPORT = False: documents not written")

**What a modeller receives**

`data/clustering_ready/` is self-contained. A teammate (or their AI assistant) starts with `README.md` (status, quick start, rules, open decisions, caveats, and a section written for AI assistants) and `BACKGROUND.md` (how the data was prepared and what is still unresolved). `decision_log.csv` holds the reason and evidence behind every choice and `transform_recipe.json` the exact parameters. Both documents are generated from this notebook's variables (cell 4-11), so rerun the notebook instead of editing them by hand.

- `X_A_scaled` / `X_B_scaled` are ready for distance-based methods; `X_A` / `X_B` are for methods that scale internally. Both are indexed by `kode8`, so any result can be joined back to `village_ids` and `interp`.
- Use `interp` (raw shares, flood, RDTR, years, geography) to profile clusters *after* fitting. Do not feed it back in without reading decision D12.
- `place_later` lists the villages outside the fit sets and why. Villages with no parcels have no tenure profile, excluded rows have unreliable areas, and the villages that only lack a registration year are already in `X_A`.
- `decision_log` and `RECIPE` document how the matrix was made and let it be rebuilt exactly.

**What the first look shows.** The PCA picture (4-8b) is a continuous cloud with two dense regions, not clearly separated groups. Whether the data support discrete clusters, or a gradient that an algorithm will cut into segments, is a question for the modelling step; cluster-tendency tests belong there.

**Known limits.** The flood zeros are unresolved, so flood stays interpretation-only. The `kab_key` still splits Kab. Bogor over two codes. The 5-unit weight on the composition block is a principled default, not the only defensible one: evidence 4-7 shows how much the alternatives change the geometry.